# MechFaithBench: быстрая проверка всех пунктов задачи

Ноутбук для Kaggle, **GPU T4 x2, Internet On**. Датасет MechFaithBench должен быть подключён через Add Input.

Что проверяется. Для любой пары выполняется точное равенство

`ER = ½ · (запас − вред контроля − утечка)`,

где *запас* — разница вероятности «да» между двумя чистыми картинками пары, *вред контроля* — сколько этой разницы теряется при
закрашивании постороннего места, *утечка* — сколько разницы остаётся, когда нужная область закрашена.
Гипотеза: ER в основном повторяет уверенность модели. Правила проверки записаны заранее в `configs/prereg.yaml`.

Два этапа на одной и той же подвыборке (50 пар на категорию):

| Этап | Что считается | Время |
|---|---|---|
| `core` | три модели в эталонных настройках; предсказание ER по чистому проходу сохраняется до закрашивания | 2–2,5 ч |
| `variations` | заливки, области, контроли и 8 контролей вместо 4 для двух меньших моделей | ещё 2,5–3 ч |

Времена — оценка; после чистого прохода каждой модели печатается прогноз. Результаты сохраняются по ходу и раз в 15 минут
упаковываются в zip. Чтобы продолжить прерванный запуск, подключите архив прошлого запуска через Add Input.

In [1]:
import os, re, sys, json, time, glob, zipfile, shutil, subprocess

SMOKE = False                    # True: 3 pairs per category through every step, to check that everything runs
STAGES = ["core", "variations"]  # leave only "core" to stop after the first stage
MODELS = [("qwen2vl_2b", 2), ("qwen25vl_3b", 2), ("qwen25vl_7b", 1)]   # (config in configs/models, processes); 1 = one process on both GPUs
VARIATION_MODELS = ["qwen2vl_2b", "qwen25vl_3b"]
PER_CATEGORY = 50
VARIATIONS = [                   # (operator, support, control, K): one change against (black, pair_union, growth, 4)
    ("gray", "pair_union", "growth", 4),
    ("mean", "pair_union", "growth", 4),
    ("blur", "pair_union", "growth", 4),
    ("noise", "pair_union", "growth", 4),
    ("black", "pair_envelope", "growth", 4),
    ("black", "pair_mask_dil", "growth", 4),
    ("black", "pair_union", "translate", 4),
    ("black", "pair_union", "radial", 4),
    ("black", "pair_union", "growth", 8),
]
CHECKPOINT_MINUTES = 15
RESTORE_FROM = None              # zip or folder with the outputs of a previous run; None = search /kaggle/input
# --- derived settings ---
IS_KAGGLE = os.path.exists("/kaggle/working")
FAKE = bool(os.environ.get("MFROB_FAKE_MODEL"))          # dry run of the pipeline without weights; never on Kaggle
BASE = "/kaggle/working" if IS_KAGGLE else os.path.abspath(os.environ.get("MFROB_SIM_DIR", "kaggle_sim"))
REPO = os.path.join(BASE, "mfrob_quick_smoke" if SMOKE else "mfrob_quick")
ZIP_NAME = "mfrob_quick_outputs_smoke.zip" if SMOKE else "mfrob_quick_outputs.zip"
ZIP_PATH = os.path.join(BASE, ZIP_NAME)
N_PER_CATEGORY = 3 if SMOKE else PER_CATEGORY
os.makedirs(REPO, exist_ok=True)

import torch
N_GPUS = torch.cuda.device_count()
print("repo:", REPO, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)] or "none")
if IS_KAGGLE:
    get_ipython().system('pip install -q -U "transformers>=4.49.0" "accelerate>=0.26.0"')

repo: /kaggle/working/mfrob_quick | GPUs: ['Tesla T4', 'Tesla T4']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 34.0 MB/s eta 0:00:00


In [2]:
CODE_VERSION = "33a23d3"
FILES = {"src/mfrob/__init__.py": "\"\"\"Robustness of the MechFaithBench behavioural protocol.\"\"\"\n", "src/mfrob/analysis/__init__.py": "", "src/mfrob/data/__init__.py": "", "src/mfrob/data/release.py": "\"\"\"Loading the MechFaithBench v1.9 release into Record objects.\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom dataclasses import dataclass, field\nfrom functools import lru_cache\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport yaml\nfrom PIL import Image\n\nSIDES = (\"original\", \"edited\")\nCATEGORY_SHORT = {\n    \"add\": \"add\", \"remove\": \"remove\", \"color_alter\": \"color\", \"counting\": \"counting\", \"relation\": \"relation\",\n    \"replace_appearance\": \"appearance\", \"textual_change\": \"text\", \"surface_material_change\": \"material\",\n}\nCATEGORY_ORDER = [\"add\", \"remove\", \"color\", \"counting\", \"relation\", \"appearance\", \"text\", \"material\"]\n\n\n@dataclass\nclass SideEvidence:\n    absent: bool                     # absence_by_design: the object is not on this side\n    mask_path: Path | None           # evidence mask; None if absent\n    boxes: list[tuple[int, int, int, int]]   # evidence component boxes, half-open xyxy; [] if absent\n    bbox: tuple[int, int, int, int] | None   # envelope of the evidence mask\n    area: int                        # evidence mask area in pixels; 0 if absent\n\n\n@dataclass\nclass Record:\n    record_id: str\n    category: str                    # release name, e.g. \"replace_appearance\"\n    group_id: str                    # sha256 of the original image; records sharing an image share it\n    image_path: dict[str, Path]\n    size: tuple[int, int] | None     # (W, H), identical on both sides; None in meta-only mode\n    question: str\n    gold: dict[str, str]             # side -> \"yes\" | \"no\"\n    evidence: dict[str, SideEvidence]\n    envelope: tuple[int, int, int, int]      # roi_bbox_x1..y2, the cross-side union envelope\n    strict_eligible: bool\n    protocol: str\n    aux_questions: dict[str, str] = field(default_factory=dict)\n\n    @property\n    def cat(self) -> str:\n        return CATEGORY_SHORT[self.category]\n\n\n_FLAT_FILL_COLOURS = re.compile(r\"\\b(gray|grey|silver|black)\\b\", re.I)\n\n\ndef fill_conflict(rec: Record) -> bool:\n    \"\"\"A colour question about a colour that a flat black, gray or mean-colour fill can produce by itself.\"\"\"\n    return rec.cat == \"color\" and bool(_FLAT_FILL_COLOURS.search(rec.question))\n\n\ndef find_release_root(config_path: str | Path = \"configs/data.yaml\") -> Path:\n    \"\"\"First configured release path that exists, else the folder of the first annotations.csv found.\"\"\"\n    cfg = yaml.safe_load(Path(config_path).read_text())\n    for p in cfg.get(\"release_roots\", []):\n        if (Path(p) / \"annotations.csv\").exists():\n            return Path(p)\n    for root in cfg.get(\"search_roots\", []):\n        hits = sorted(Path(root).rglob(\"annotations.csv\")) if Path(root).is_dir() else []\n        if hits:\n            return hits[0].parent\n    raise FileNotFoundError(\"MechFaithBench release not found; see configs/data.yaml and scripts/download_release.py\")\n\n\ndef read_checksums(root: Path) -> dict[str, str]:\n    out = {}\n    for line in (root / \"checksums.sha256\").read_text().splitlines():\n        if line.strip():\n            digest, path = line.split(None, 1)\n            out[path.lstrip(\"*\").strip()] = digest\n    return out\n\n\ndef _box(b) -> tuple[int, int, int, int] | None:\n    return None if b is None else tuple(int(v) for v in b)\n\n\ndef _side_evidence(root: Path, side: dict) -> SideEvidence:\n    # Nine absent sides in `counting` have no evidence_* keys at all, hence .get everywhere.\n    mask = side.get(\"evidence_mask\")\n    return SideEvidence(\n        absent=bool(side.get(\"absence_by_design\", False)),\n        mask_path=(root / mask) if mask else None,\n        boxes=[_box(b) for b in (side.get(\"evidence_component_bboxes_xyxy\") or [])],\n        bbox=_box(side.get(\"evidence_bbox_xyxy\")),\n        area=int(side.get(\"evidence_area_pixels\") or 0),\n    )\n\n\ndef image_size(path: Path) -> tuple[int, int]:\n    with Image.open(path) as im:\n        return im.size\n\n\ndef load_release(root: str | Path, with_sizes: bool = True) -> list[Record]:\n    \"\"\"All 1,600 records. with_sizes=False reads metadata only (no image files needed).\"\"\"\n    root = Path(root)\n    ann = pd.read_csv(root / \"annotations.csv\", keep_default_na=False)\n    roi = {r[\"record_id\"]: r for r in json.loads((root / \"roi_annotations.json\").read_text())[\"records\"]}\n    checksums = read_checksums(root)\n    records = []\n    for row in ann.itertuples():\n        r = roi[row.record_id]\n        paths = {\"original\": root / row.original_image, \"edited\": root / row.edited_image}\n        size = None\n        if with_sizes:\n            size = image_size(paths[\"original\"])\n            other = image_size(paths[\"edited\"])\n            if other != size:\n                raise ValueError(f\"{row.record_id}: sides differ in size, {size} vs {other}\")\n        aux = {}\n        for d in json.loads(row.diagnostic_questions_json or \"[]\"):\n            if d.get(\"type\") == \"non_flip_control\":\n                aux[\"non_flip\"] = d[\"question\"]\n        for k in (1, 2, 3):\n            aux[f\"probe{k}\"] = getattr(row, f\"hallucination_probe_{k}\")\n        records.append(Record(\n            record_id=row.record_id,\n            category=row.category,                         # never derive it from the id prefix or the folder\n            group_id=checksums[row.original_image],\n            image_path=paths,\n            size=size,\n            question=row.question,\n            gold={\"original\": row.gold_original, \"edited\": row.gold_edited},\n            evidence={s: _side_evidence(root, r[s]) for s in SIDES},\n            envelope=(int(row.roi_bbox_x1), int(row.roi_bbox_y1), int(row.roi_bbox_x2), int(row.roi_bbox_y2)),\n            strict_eligible=row.roi_strict_spatial_eligible == \"yes\",\n            protocol=row.roi_evidence_protocol,\n            aux_questions=aux,\n        ))\n    return records\n\n\n@lru_cache(maxsize=4096)\ndef load_mask(path: Path) -> np.ndarray:\n    \"\"\"Evidence mask as a bool array (H, W); cached.\"\"\"\n    return np.asarray(Image.open(path).convert(\"L\")) > 127\n", "src/mfrob/data/subset.py": "\"\"\"A fixed random subset of the release, shared by every model (PLAN.md, stage 6.1).\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom ..geometry.controls import growth_feasible, seed_from\nfrom ..geometry.supports import forbidden_zone, pair_union\nfrom .release import Record\n\n\ndef has_control(rec: Record) -> bool:\n    \"\"\"A growth control exists for the pair_union region.\"\"\"\n    u = pair_union(rec)\n    return growth_feasible(forbidden_zone(u, u), int(u.sum()))\n\n\ndef subset(records: list[Record], per_category: int, seed: int = 0) -> list[Record]:\n    \"\"\"The first `per_category` candidates of every category in a seeded permutation, in release order.\n\n    Candidates: strictly eligible records for which a control exists; this does not depend on any model.\n    Records that share an original image with a chosen record are added, so image groups are never split.\n    \"\"\"\n    cands = [r for r in records if r.strict_eligible and has_control(r)]\n    chosen = set()\n    for cat in sorted({r.category for r in cands}):\n        ids = sorted(r.record_id for r in cands if r.category == cat)\n        order = np.random.default_rng(seed_from(\"subset\", seed, cat)).permutation(len(ids))\n        chosen.update(ids[i] for i in order[:per_category])\n    groups = {r.group_id for r in cands if r.record_id in chosen}\n    return [r for r in cands if r.group_id in groups]\n", "src/mfrob/geometry/__init__.py": "", "src/mfrob/geometry/controls.py": "\"\"\"Control regions: same pixel count as the support, away from the evidence.\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\n\nimport numpy as np\nfrom scipy import ndimage, signal\n\nFOUR = ndimage.generate_binary_structure(2, 1)     # 4-connectivity\nRADIAL_NEAREST = 200                               # `radial` draws from this many shifts closest in eccentricity\n\n\ndef seed_from(*parts) -> int:\n    \"\"\"Reproducible seed; Python's hash() is salted per process and must not be used.\"\"\"\n    return int.from_bytes(hashlib.sha256(\"|\".join(map(str, parts)).encode()).digest()[:8], \"big\")\n\n\ndef big_enough_components(forbidden: np.ndarray, n: int) -> np.ndarray:\n    \"\"\"Pixels of the free 4-connected components that hold at least n pixels.\"\"\"\n    lab, k = ndimage.label(~forbidden, structure=FOUR)\n    if k == 0:\n        return np.zeros_like(forbidden)\n    sizes = np.bincount(lab.ravel())\n    sizes[0] = 0\n    return np.isin(lab, np.nonzero(sizes >= n)[0])\n\n\ndef growth_feasible(forbidden: np.ndarray, n: int) -> bool:\n    return bool(big_enough_components(forbidden, n).any())\n\n\ndef growth(forbidden: np.ndarray, n: int, seed: int) -> np.ndarray | None:\n    \"\"\"Exactly n pixels grown breadth-first (4-connectivity) from a random free seed; None if infeasible.\n    Whole layers are added in order; the last layer is filled with a random subset.\"\"\"\n    ok = big_enough_components(forbidden, n)\n    if not ok.any():\n        return None\n    rng = np.random.default_rng(seed)\n    h, w = forbidden.shape\n    ys, xs = np.nonzero(ok)\n    j = int(rng.integers(len(ys)))\n    cur = np.zeros((h, w), bool)\n    cur[ys[j], xs[j]] = True\n    count, y0, y1, x0, x1 = 1, ys[j], ys[j] + 1, xs[j], xs[j] + 1\n    while count < n:\n        # work on the bounding window of the current blob, grown by one pixel\n        y0, y1, x0, x1 = max(0, y0 - 1), min(h, y1 + 1), max(0, x0 - 1), min(w, x1 + 1)\n        c, free = cur[y0:y1, x0:x1], ok[y0:y1, x0:x1]\n        nb = np.zeros_like(c)\n        nb[1:] |= c[:-1]; nb[:-1] |= c[1:]; nb[:, 1:] |= c[:, :-1]; nb[:, :-1] |= c[:, 1:]\n        frontier = nb & free & ~c\n        k = int(frontier.sum())\n        if k == 0:                                   # cannot happen inside a component with >= n pixels\n            return None\n        if count + k <= n:\n            c |= frontier\n            count += k\n        else:\n            fy, fx = np.nonzero(frontier)\n            pick = rng.choice(k, n - count, replace=False)\n            c[fy[pick], fx[pick]] = True\n            count = n\n    return cur\n\n\ndef allowed_shifts(shape: np.ndarray, forbidden: np.ndarray) -> tuple[np.ndarray, tuple[int, int, int, int]]:\n    \"\"\"Where the shape can be moved to as a rigid figure.\n\n    Returns (ok, (y0, y1, x0, x1)): the half-open bounding box of the shape, and ok[ty, tx] = True if the shape,\n    moved so that this box starts at (ty, tx), lies inside the image and covers no forbidden pixel.\n    Exact: the number of covered forbidden pixels is a cross-correlation, computed by FFT.\n    \"\"\"\n    ys, xs = np.nonzero(shape)\n    y0, y1, x0, x1 = int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1\n    kern = shape[y0:y1, x0:x1].astype(np.float64)\n    overlap = signal.fftconvolve(forbidden.astype(np.float64), kern[::-1, ::-1], mode=\"valid\")\n    return overlap < 0.5, (y0, y1, x0, x1)\n\n\ndef translate(shape: np.ndarray, forbidden: np.ndarray, k: int, seed: int, nearest: int | None = None) -> list[np.ndarray] | None:\n    \"\"\"k distinct rigid shifts of the shape that avoid `forbidden`; None if fewer than k exist.\n\n    nearest=None: drawn uniformly from all allowed shifts (the `translate` control).\n    nearest=n: drawn uniformly from the n allowed shifts whose eccentricity is closest to the shape's (`radial`).\n    \"\"\"\n    ok, (y0, y1, x0, x1) = allowed_shifts(shape, forbidden)\n    tys, txs = np.nonzero(ok)\n    if len(tys) < k:\n        return None\n    if nearest is not None:\n        h, w = shape.shape\n        cy, cx = (h - 1) / 2, (w - 1) / 2\n        ecc = lambda ty, tx: np.hypot(ty + (y1 - y0 - 1) / 2 - cy, tx + (x1 - x0 - 1) / 2 - cx) / np.hypot(cy, cx)   # noqa: E731\n        order = np.argsort(np.abs(ecc(tys, txs) - ecc(y0, x0)), kind=\"stable\")[:max(nearest, k)]\n        tys, txs = tys[order], txs[order]\n    kern = shape[y0:y1, x0:x1]\n    out = []\n    for j in np.random.default_rng(seed).choice(len(tys), k, replace=False):\n        m = np.zeros_like(shape)\n        m[tys[j]:tys[j] + (y1 - y0), txs[j]:txs[j] + (x1 - x0)] = kern\n        out.append(m)\n    return out\n", "src/mfrob/geometry/supports.py": "\"\"\"Occlusion regions (supports) as bool masks of shape (H, W).\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\n\nimport numpy as np\nfrom scipy import ndimage\n\nfrom ..data.release import SIDES, Record, load_mask\n\nMARGIN = 8          # px between a control and any evidence; see PLAN.md, section 3.4\n\n\ndef rasterize(boxes, size: tuple[int, int]) -> np.ndarray:\n    \"\"\"Union of half-open xyxy boxes; size is (W, H).\"\"\"\n    w, h = size\n    m = np.zeros((h, w), bool)\n    for x1, y1, x2, y2 in boxes:\n        m[max(0, y1):min(h, y2), max(0, x1):min(w, x2)] = True\n    return m\n\n\ndef pair_union(rec: Record) -> np.ndarray:\n    \"\"\"Union of the evidence component boxes of both sides; the same region on both sides (reference support).\"\"\"\n    return rasterize([b for s in SIDES for b in rec.evidence[s].boxes], rec.size)\n\n\ndef pair_envelope(rec: Record) -> np.ndarray:\n    \"\"\"The single cross-side envelope box roi_bbox_x1..y2.\"\"\"\n    return rasterize([rec.envelope], rec.size)\n\n\ndef pair_mask_dil(rec: Record) -> np.ndarray:\n    \"\"\"Union of the evidence masks of both sides, dilated by MARGIN px; the same region on both sides.\"\"\"\n    masks = [load_mask(rec.evidence[s].mask_path) for s in SIDES if rec.evidence[s].mask_path is not None]\n    return dilate(np.logical_or.reduce(masks), MARGIN)\n\n\nSUPPORTS = {\"pair_union\": pair_union, \"pair_envelope\": pair_envelope, \"pair_mask_dil\": pair_mask_dil}\n\n\ndef dilate(mask: np.ndarray, r: int) -> np.ndarray:\n    \"\"\"Dilation by a (2r+1) x (2r+1) square, clipped to the image.\"\"\"\n    if r <= 0:\n        return mask\n    return ndimage.maximum_filter(mask.astype(np.uint8), size=2 * r + 1, mode=\"constant\", cval=0).astype(bool)\n\n\ndef forbidden_zone(support: np.ndarray, union: np.ndarray, margin: int = MARGIN) -> np.ndarray:\n    \"\"\"Where a control may not go: the support and all annotated evidence of both sides, plus a margin.\"\"\"\n    return dilate(support | union, margin)\n\n\ndef mask_hash(mask: np.ndarray) -> str:\n    return hashlib.sha1(np.packbits(mask).tobytes() + str(mask.shape).encode()).hexdigest()\n\n\ndef eccentricity(mask: np.ndarray) -> float:\n    \"\"\"Distance from the centre of the mask's bounding rectangle to the image centre, in half-diagonals.\"\"\"\n    ys, xs = np.nonzero(mask)\n    h, w = mask.shape\n    cy, cx = (h - 1) / 2, (w - 1) / 2\n    return float(np.hypot((ys.min() + ys.max()) / 2 - cy, (xs.min() + xs.max()) / 2 - cx) / np.hypot(cy, cx))\n", "src/mfrob/metrics/__init__.py": "", "src/mfrob/metrics/bootstrap.py": "\"\"\"Cluster bootstrap over image groups: percentile intervals for means, ratios of sums and macro means.\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nN_BOOT, SEED, LEVEL = 10_000, 20260803, 0.95\n\n\ndef _draws(groups, n_boot: int, seed: int):\n    codes, _ = pd.factorize(np.asarray(groups))\n    n = int(codes.max()) + 1\n    return codes, n, np.random.default_rng(seed).integers(0, n, (n_boot, n), dtype=np.int32)\n\n\ndef _ci(point: float, samples: np.ndarray, level: float) -> tuple[float, float, float]:\n    a = 100 * (1 - level) / 2\n    return float(point), float(np.nanpercentile(samples, a)), float(np.nanpercentile(samples, 100 - a))\n\n\ndef boot_mean(x, groups, n_boot: int = N_BOOT, seed: int = SEED, level: float = LEVEL) -> tuple[float, float, float]:\n    \"\"\"Mean of x with a percentile interval; whole groups are resampled.\"\"\"\n    x = np.asarray(x, float)\n    codes, n, idx = _draws(groups, n_boot, seed)\n    sums, cnt = np.bincount(codes, weights=x, minlength=n), np.bincount(codes, minlength=n)\n    return _ci(x.mean(), sums[idx].sum(1) / cnt[idx].sum(1), level)\n\n\ndef boot_ratio(num, den, groups, n_boot: int = N_BOOT, seed: int = SEED, level: float = LEVEL) -> tuple[float, float, float]:\n    \"\"\"sum(num) / sum(den) with a percentile interval.\"\"\"\n    num, den = np.asarray(num, float), np.asarray(den, float)\n    codes, n, idx = _draws(groups, n_boot, seed)\n    a, b = np.bincount(codes, weights=num, minlength=n), np.bincount(codes, weights=den, minlength=n)\n    return _ci(num.sum() / den.sum(), a[idx].sum(1) / b[idx].sum(1), level)\n\n\ndef boot_macro(x, cats, groups, n_boot: int = N_BOOT, seed: int = SEED, level: float = LEVEL) -> tuple[float, float, float]:\n    \"\"\"Mean of the category means of x; categories that are empty in a resample are left out of its mean.\"\"\"\n    x = np.asarray(x, float)\n    codes, n, idx = _draws(groups, n_boot, seed)\n    ccodes, names = pd.factorize(np.asarray(cats))\n    means = []\n    for j in range(len(names)):\n        m = ccodes == j\n        s, c = np.bincount(codes[m], weights=x[m], minlength=n), np.bincount(codes[m], minlength=n)\n        tot = c[idx].sum(1)\n        means.append(np.where(tot > 0, s[idx].sum(1) / np.maximum(tot, 1), np.nan))\n    point = float(np.mean([x[ccodes == j].mean() for j in range(len(names))]))\n    return _ci(point, np.nanmean(np.vstack(means), axis=0), level)\n", "src/mfrob/metrics/er.py": "\"\"\"Evidence Reliance, its parts and its variants on other scales, from saved logits.\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ..data.release import SIDES\n\n\nLD_CLIP = 20.0          # log-odds are clipped to +-LD_CLIP before any log-odds statistic\nREL_MIN = 0.05          # the relative scale is undefined where the control probability is below this\n\n\ndef sigmoid(x):\n    return 1.0 / (1.0 + np.exp(-np.asarray(x, float)))\n\n\ndef pair_er(interv: pd.DataFrame, gold: dict[tuple[str, str], str]) -> pd.DataFrame:\n    \"\"\"One row per pair with target + controls: ER, its YES/NO parts and ROA. Negative values are kept.\n\n    gold maps (record_id, side) -> \"yes\" | \"no\". p(gold) = sigmoid(+ld) on a yes side, sigmoid(-ld) on a no side.\n    ER_side = mean_k p(gold | control_k) - p(gold | target);  ER = (ER_yes + ER_no) / 2.\n    ROA = 1 if the summed gold probability under the target is below that under every control.\n    p_target_*, p_control_*: p(gold) on the yes / no side.  wrong_target_*: the answer under the target is not gold;\n    wrong_control_*: share of the controls under which it is not gold.\n    \"\"\"\n    d = interv.drop_duplicates([\"record_id\", \"side\", \"region_kind\", \"k\"], keep=\"last\").copy()\n    sign = np.array([1.0 if gold[(r, s)] == \"yes\" else -1.0 for r, s in zip(d.record_id, d.side)])\n    d[\"p_gold\"] = sigmoid(sign * d.ld.values)\n    d[\"gold\"] = np.where(sign > 0, \"yes\", \"no\")\n    rows = []\n    for rid, g in d.groupby(\"record_id\", sort=False):\n        t = g[g.region_kind == \"target\"].set_index(\"side\").p_gold\n        c = g[g.region_kind == \"control\"]\n        if len(t) != 2 or c.empty:\n            continue\n        cm = c.groupby(\"side\").p_gold.mean()\n        cw = (c.p_gold < 0.5).groupby(c.side).mean()\n        side_of = {v: k for k, v in g.drop_duplicates(\"side\").set_index(\"side\").gold.items()}      # \"yes\" -> side\n        per_k = c.pivot(index=\"k\", columns=\"side\", values=\"p_gold\").sum(axis=1)\n        row = {\"record_id\": rid, \"roa\": bool(float(t.sum()) < float(per_k.min())), \"k\": int(per_k.shape[0]),\n               \"n_pixels\": int(g.n_pixels.iloc[0])}\n        for a, side in side_of.items():\n            row.update({f\"er_{a}\": float(cm[side] - t[side]), f\"p_target_{a}\": float(t[side]), f\"p_control_{a}\": float(cm[side]),\n                        f\"wrong_target_{a}\": bool(t[side] < 0.5), f\"wrong_control_{a}\": float(cw[side])})\n        row[\"er\"] = (row[\"er_yes\"] + row[\"er_no\"]) / 2\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef summarize(er: pd.DataFrame, pairs: pd.DataFrame, n_eligible: int | None = None) -> dict:\n    \"\"\"Micro means over pairs. pairs: clean-pass table with B per record. LVF = coverage x ER over the eligible set.\"\"\"\n    out = {\"pairs\": int(len(er)), \"er\": float(er.er.mean()), \"er_yes\": float(er.er_yes.mean()),\n           \"er_no\": float(er.er_no.mean()), \"roa\": float(er.roa.mean())}\n    if n_eligible:\n        out[\"coverage\"] = len(er) / n_eligible\n        out[\"lvf\"] = out[\"coverage\"] * out[\"er\"]\n    return out\n\n\ndef components(interv: pd.DataFrame, pairs: pd.DataFrame, gold: dict[tuple[str, str], str],\n               ecc_gap: float | None = None) -> pd.DataFrame:\n    \"\"\"One row per pair with a target and controls on both sides: the parts of ER, and ER on four scales.\n\n    interv: rows of one experiment. pairs: clean-pass table indexed by record_id (columns ld_yes, ld_no).\n    P = P(yes) = sigmoid(ld); Y / N = the side whose gold answer is yes / no; C_k = control k; T = target occluded.\n\n    margin   = Y_clean - N_clean                  discrimination of the pair without any intervention\n    harm     = margin - mean_k (Y_Ck - N_Ck)      what an unrelated occlusion costs\n    leak     = Y_T - N_T                          discrimination left with the evidence region occluded\n    default  = (Y_T + N_T) / 2                    the answer when the evidence is hidden\n    er       = (margin - harm - leak) / 2         an exact identity; equals pair_er(...).er\n    ps, roa  = share of the controls with (Y_Ck - N_Ck) > leak, and whether all of them are\n    ld_margin, ld_harm, ld_leak                   the same three parts on the log-odds scale\n    er_S, er_yes_S, er_no_S                       S in prob, logodds, rel, flip (definitions in configs/prereg.yaml)\n    er_matched, n_matched                         ER from the controls whose eccentricity is within ecc_gap of the target's\n    \"\"\"\n    d = interv.drop_duplicates([\"record_id\", \"side\", \"region_kind\", \"k\"], keep=\"last\")\n    d = d.assign(gold=[gold[(r, s)] for r, s in zip(d.record_id, d.side)], ld=d.ld.clip(-LD_CLIP, LD_CLIP))\n    tg, ct = d[d.region_kind == \"target\"], d[d.region_kind == \"control\"]\n    t = tg.pivot(index=\"record_id\", columns=\"gold\", values=\"ld\").reindex(columns=[\"yes\", \"no\"]).dropna()\n    c = ct.pivot(index=[\"record_id\", \"k\"], columns=\"gold\", values=\"ld\").reindex(columns=[\"yes\", \"no\"]).dropna()\n    c = c[c.index.get_level_values(\"record_id\").isin(t.index)]\n    rec = c.index.get_level_values(\"record_id\")\n    t = t.loc[t.index.isin(rec)]\n    prob = lambda x: pd.Series(sigmoid(x.values), index=x.index)                      # noqa: E731\n    ty, tn = prob(t.yes), prob(t.no)                             # P(yes) under the target, per pair\n    cy, cn = prob(c.yes), prob(c.no)                             # P(yes) under each control\n    leak, ld_leak = ty - tn, t.yes - t.no\n    dk, ld_dk = cy - cn, c.yes - c.no                            # discrimination under each control\n    by = lambda x: x.groupby(rec, sort=False).mean().reindex(t.index)                 # noqa: E731\n    out = pd.DataFrame(index=t.index)\n    out[\"k\"] = c.groupby(rec, sort=False).size().reindex(t.index)\n    out[\"n_pixels\"] = tg.groupby(\"record_id\").n_pixels.first().reindex(t.index)\n    P = pairs.reindex(t.index)\n    cl_y, cl_n = P.ld_yes.clip(-LD_CLIP, LD_CLIP), P.ld_no.clip(-LD_CLIP, LD_CLIP)\n    out[\"margin\"] = prob(cl_y) - prob(cl_n)\n    out[\"harm\"] = out.margin - by(dk)\n    out[\"leak\"], out[\"default\"] = leak, (ty + tn) / 2\n    out[\"er\"] = (by(dk) - leak) / 2\n    out[\"ps\"] = by((dk > leak.reindex(rec).values).astype(float))\n    out[\"roa\"] = out.ps == 1.0\n    out[\"ld_margin\"], out[\"ld_leak\"] = cl_y - cl_n, ld_leak\n    out[\"ld_harm\"] = out.ld_margin - by(ld_dk)\n    out[\"roa_logodds\"] = by((ld_dk > ld_leak.reindex(rec).values).astype(float)) == 1.0\n    # scales; on a no side p(gold) = 1 - P(yes)\n    out[\"er_yes_prob\"], out[\"er_no_prob\"] = by(cy) - ty, tn - by(cn)\n    out[\"er_yes_logodds\"], out[\"er_no_logodds\"] = by(c.yes) - t.yes, t.no - by(c.no)\n    pc_y, pc_n = by(cy), 1 - by(cn)\n    out[\"er_yes_rel\"] = ((pc_y - ty) / pc_y).where(pc_y >= REL_MIN)\n    out[\"er_no_rel\"] = ((pc_n - (1 - tn)) / pc_n).where(pc_n >= REL_MIN)\n    out[\"er_yes_flip\"] = (ty < 0.5).astype(float) - by((cy < 0.5).astype(float))\n    out[\"er_no_flip\"] = (tn > 0.5).astype(float) - by((cn > 0.5).astype(float))\n    for s in (\"prob\", \"logodds\", \"rel\", \"flip\"):\n        out[f\"er_{s}\"] = (out[f\"er_yes_{s}\"] + out[f\"er_no_{s}\"]) / 2\n    out[\"er_yes\"], out[\"er_no\"] = out.er_yes_prob, out.er_no_prob\n    if ecc_gap is not None:\n        te = tg.groupby(\"record_id\").eccentricity.first()\n        ce = ct.groupby([\"record_id\", \"k\"]).eccentricity.first().reindex(c.index)\n        near = (ce - te.reindex(rec).values).abs() <= ecc_gap\n        out[\"n_matched\"] = near.groupby(rec, sort=False).sum().reindex(t.index)\n        out[\"er_matched\"] = (dk[near].groupby(rec[near], sort=False).mean().reindex(t.index) - leak) / 2\n    return out\n", "src/mfrob/models/__init__.py": "\"\"\"Model registry.\"\"\"\nfrom __future__ import annotations\n\nimport os\n\n\ndef make_model(cfg: dict):\n    \"\"\"The model of a config file. MFROB_FAKE_MODEL=1 gives a tiny stand-in for dry runs of the pipeline without weights.\"\"\"\n    if os.environ.get(\"MFROB_FAKE_MODEL\"):\n        from .fake import FakeVLM\n        return FakeVLM(cfg)\n    from .qwen_vl import QwenVL\n    return QwenVL(cfg)\n", "src/mfrob/models/base.py": "\"\"\"Common interface of the scored vision-language models.\"\"\"\nfrom __future__ import annotations\n\nfrom abc import ABC, abstractmethod\n\nimport torch\nfrom PIL import Image\n\nYES_SPELLINGS = [\"Yes\", \"yes\", \" Yes\", \" yes\"]\nNO_SPELLINGS = [\"No\", \"no\", \" No\", \" no\"]\n\n\ndef pick_device() -> str:\n    if torch.cuda.is_available():\n        return \"cuda\"\n    if torch.backends.mps.is_available():\n        return \"mps\"\n    return \"cpu\"\n\n\ndef pick_dtype(device: str, requested: str = \"auto\") -> torch.dtype:\n    \"\"\"auto: bf16 on Ampere+ GPUs, fp16 on older CUDA GPUs, fp32 on mps and cpu.\"\"\"\n    if requested != \"auto\":\n        return getattr(torch, requested)\n    if device == \"cuda\":\n        return torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16\n    return torch.float32\n\n\ndef single_token_ids(tokenizer, spellings: list[str]) -> list[int]:\n    \"\"\"Ids of the spellings that the tokenizer encodes as exactly one token.\"\"\"\n    ids = set()\n    for w in spellings:\n        enc = tokenizer.encode(w, add_special_tokens=False)\n        if len(enc) == 1:\n            ids.add(enc[0])\n    return sorted(ids)\n\n\ndef score_from_logits(logits: torch.Tensor, yes_ids: list[int], no_ids: list[int]) -> dict:\n    \"\"\"Yes/no summary of next-token logits (1-D, any dtype). Everything is computed in fp32.\"\"\"\n    l = logits.float()\n    if not torch.isfinite(l).all():\n        raise FloatingPointError(\"non-finite logits\")\n    l_yes, l_no = torch.logsumexp(l[yes_ids], 0), torch.logsumexp(l[no_ids], 0)\n    top1 = int(l.argmax())\n    return {\n        \"ld\": float(l_yes - l_no), \"l_yes\": float(l_yes), \"l_no\": float(l_no),\n        \"logits_yes\": [float(v) for v in l[yes_ids]], \"logits_no\": [float(v) for v in l[no_ids]],\n        \"lse_all\": float(torch.logsumexp(l, 0)), \"top1_id\": top1, \"top1_is_yesno\": top1 in yes_ids or top1 in no_ids,\n    }\n\n\nclass VLM(ABC):\n    name: str\n    hf_id: str\n    revision: str | None\n    device: str\n    dtype: torch.dtype\n    max_pixels: int\n    prompt_template: str\n\n    @abstractmethod\n    def yes_no_ids(self) -> tuple[list[int], list[int]]: ...\n\n    @abstractmethod\n    def score(self, images: list[Image.Image], question: str) -> list[dict]:\n        \"\"\"One forward pass without generation; one dict per image (see score_from_logits) plus n_img_tokens.\"\"\"\n", "src/mfrob/models/fake.py": "\"\"\"A stand-in model for dry runs: no weights, deterministic scores that react to occlusion. Never use its numbers.\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\nfrom .base import VLM\n\n\nclass FakeVLM(VLM):\n    def __init__(self, cfg: dict):\n        self.name, self.hf_id, self.max_pixels = cfg[\"name\"], \"fake\", int(cfg[\"max_pixels\"])\n        self.revision, self.device, self.dtype, self.prompt_template = \"fake\", \"cpu\", torch.float32, \"{question}\"\n\n    def yes_no_ids(self) -> tuple[list[int], list[int]]:\n        return [1], [0]\n\n    def score(self, images: list[Image.Image], question: str) -> list[dict]:\n        assert len({im.size for im in images}) == 1, \"images of one batch must share a size\"\n        out = []\n        for im in images:\n            h = hashlib.sha1(np.asarray(im.convert(\"RGB\")).tobytes() + question.encode() + self.name.encode()).digest()\n            ld = (int.from_bytes(h[:4], \"big\") / 2**32 - 0.5) * 8\n            out.append({\"ld\": ld, \"l_yes\": ld, \"l_no\": 0.0, \"logits_yes\": [ld], \"logits_no\": [0.0], \"lse_all\": abs(ld),\n                        \"top1_id\": int(ld > 0), \"top1_is_yesno\": True, \"n_img_tokens\": 4})\n        return out\n", "src/mfrob/models/qwen_vl.py": "\"\"\"Qwen2-VL / Qwen2.5-VL scoring.\"\"\"\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport torch\nfrom PIL import Image\nfrom transformers import AutoModelForImageTextToText, AutoProcessor\n\nfrom .base import NO_SPELLINGS, VLM, YES_SPELLINGS, pick_device, pick_dtype, score_from_logits, single_token_ids\n\n\nclass QwenVL(VLM):\n    def __init__(self, cfg: dict):\n        self.name, self.hf_id, self.max_pixels = cfg[\"name\"], cfg[\"hf_id\"], int(cfg[\"max_pixels\"])\n        self.device = cfg.get(\"device\") or pick_device()\n        self.dtype = pick_dtype(self.device, cfg.get(\"dtype\", \"auto\"))\n        kw = {\"device_map\": \"auto\"} if self.device == \"cuda\" else {}\n        self.model = AutoModelForImageTextToText.from_pretrained(self.hf_id, dtype=self.dtype, **kw).eval()\n        if self.device != \"cuda\":\n            self.model.to(self.device)\n        # The resolution cap is passed with every call as `size`; the constructor argument `max_pixels` is deprecated\n        # in transformers 5 and would be dropped silently by a later version.\n        self.processor = AutoProcessor.from_pretrained(self.hf_id)\n        ip = self.processor.image_processor\n        lo = int(getattr(ip.size, \"shortest_edge\", None) or 56 * 56)\n        self._size = {\"shortest_edge\": lo, \"longest_edge\": self.max_pixels}\n        self._size_kwargs = [{\"size\": self._size}, {\"min_pixels\": lo, \"max_pixels\": self.max_pixels}]    # new and old spelling\n        self._cell = int(getattr(ip, \"patch_size\", 14)) * int(getattr(ip, \"merge_size\", 2))      # pixels per image token side\n        self.revision = getattr(self.model.config, \"_commit_hash\", None)\n        if self.revision is None:                      # newer transformers: take it from the cache path\n            from huggingface_hub import snapshot_download\n            self.revision = Path(snapshot_download(self.hf_id, local_files_only=True)).name\n        tok = self.processor.tokenizer\n        self._yes, self._no = single_token_ids(tok, YES_SPELLINGS), single_token_ids(tok, NO_SPELLINGS)\n        assert self._yes and self._no\n        self.image_token_id = getattr(self.model.config, \"image_token_id\", None) or tok.convert_tokens_to_ids(\"<|image_pad|>\")\n        self.prompt_template = self._text(\"{question}\")\n        self._in_device = next(self.model.parameters()).device\n\n    def _text(self, question: str) -> str:\n        messages = [{\"role\": \"user\", \"content\": [{\"type\": \"image\"}, {\"type\": \"text\", \"text\": question}]}]\n        return self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n    def yes_no_ids(self) -> tuple[list[int], list[int]]:\n        return self._yes, self._no\n\n    def _encode(self, texts: list[str], images: list[Image.Image]):\n        \"\"\"Processor call with the resolution cap. transformers versions spell the cap differently and ignore an\n        unknown spelling with a warning, so every result is checked: the first spelling that keeps the number of\n        image tokens under the cap is returned. (An image below the cap passes with any spelling, harmlessly.)\"\"\"\n        limit = self.max_pixels // self._cell ** 2\n        for kw in self._size_kwargs:\n            try:\n                out = self.processor(text=texts, images=images, return_tensors=\"pt\", padding=True, **kw)\n            except (TypeError, ValueError):\n                continue\n            if int((out[\"input_ids\"] == self.image_token_id).sum(1).max()) <= limit:\n                return out\n        raise RuntimeError(f\"the resolution cap max_pixels = {self.max_pixels} could not be applied with this transformers version\")\n\n    @torch.no_grad()\n    def score(self, images: list[Image.Image], question: str) -> list[dict]:\n        # Images of one batch must have the same size: then every sequence has the same length and\n        # no padding is involved, so the last position is the answer position for all of them.\n        assert len({im.size for im in images}) == 1, \"images of one batch must share a size\"\n        inputs = self._encode([self._text(question)] * len(images), [im.convert(\"RGB\") for im in images]).to(self._in_device)\n        assert bool(inputs[\"attention_mask\"].all()), \"unexpected padding\"\n        n_img = (inputs[\"input_ids\"] == self.image_token_id).sum(1).tolist()\n        logits = self.model(**inputs, use_cache=False).logits[:, -1]\n        out = []\n        for i in range(len(images)):\n            d = score_from_logits(logits[i].cpu(), self._yes, self._no)\n            d[\"n_img_tokens\"] = int(n_img[i])\n            out.append(d)\n        return out\n", "src/mfrob/operators/__init__.py": "", "src/mfrob/operators/pixel.py": "\"\"\"Pixel-level occlusion operators. Pixels outside the region are never touched.\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nfrom PIL import Image, ImageFilter\n\nOPERATORS = (\"black\", \"gray\", \"mean\", \"blur\", \"noise\")\n\n\ndef blurred(image: Image.Image) -> np.ndarray:\n    \"\"\"The whole image after a Gaussian blur with sigma = min(W, H) / 32 (Pillow's box-blur approximation).\"\"\"\n    im = image.convert(\"RGB\")\n    return np.asarray(im.filter(ImageFilter.GaussianBlur(min(im.size) / 32)))\n\n\ndef apply(image: Image.Image, region: np.ndarray, op: str, seed: int = 0, blur: np.ndarray | None = None) -> Image.Image:\n    \"\"\"blur: precomputed blurred(image), to avoid blurring once per region.\"\"\"\n    a = np.asarray(image.convert(\"RGB\")).copy()\n    assert region.shape == a.shape[:2], (region.shape, a.shape)\n    if op == \"black\":\n        a[region] = 0\n    elif op == \"gray\":\n        a[region] = 128\n    elif op == \"mean\":\n        a[region] = np.rint(a.reshape(-1, 3).mean(0)).astype(np.uint8)\n    elif op == \"blur\":\n        a[region] = (blurred(image) if blur is None else blur)[region]\n    elif op == \"noise\":\n        a[region] = np.random.default_rng(seed).integers(0, 256, (int(region.sum()), 3), dtype=np.uint8)\n    else:\n        raise ValueError(f\"unknown operator {op}\")\n    return Image.fromarray(a)\n", "src/mfrob/run/__init__.py": "", "src/mfrob/run/clean_pass.py": "\"\"\"Clean pass: both sides of every pair, primary question, no occlusion. Resumable.\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\nfrom PIL import Image\n\nfrom ..data.release import SIDES, Record\nfrom ..models.base import VLM\n\n\ndef image_key(record_id: str, side: str, operator: str = \"clean\", region_hash: str = \"\") -> str:\n    return hashlib.sha1(f\"{record_id}|{side}|{operator}|{region_hash}\".encode()).hexdigest()\n\n\ndef load_logits(out_dir: Path) -> pd.DataFrame:\n    parts = sorted(Path(out_dir).glob(\"part-*.parquet\"))\n    return pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True) if parts else pd.DataFrame()\n\n\ndef run_clean_pass(model: VLM, records: list[Record], out_dir: Path, shard_size: int = 200, log_every: int = 50, tag: str = \"\") -> None:\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    done = load_logits(out_dir)\n    done_ids = set(done[done.question_kind == \"primary\"].record_id) if len(done) else set()\n    todo = [r for r in records if r.record_id not in done_ids]\n    print(f\"clean pass: {len(records)} pairs, {len(done_ids)} already done, {len(todo)} to go\")\n    rows, t0 = [], time.time()\n    n_parts = len(list(out_dir.glob(f\"part-{tag}*.parquet\")))\n\n    def flush():\n        nonlocal rows, n_parts\n        if rows:\n            pd.DataFrame(rows).to_parquet(out_dir / f\"part-{tag}{n_parts:05d}.parquet\", index=False)\n            n_parts += 1\n            rows = []\n\n    for i, rec in enumerate(todo, 1):\n        images = [Image.open(rec.image_path[s]) for s in SIDES]\n        scores = model.score(images, rec.question)\n        assert scores[0][\"n_img_tokens\"] == scores[1][\"n_img_tokens\"]\n        for side, sc in zip(SIDES, scores):\n            rows.append({\"model\": model.name, \"record_id\": rec.record_id, \"side\": side, \"question_kind\": \"primary\",\n                         \"image_key\": image_key(rec.record_id, side), **sc, \"dtype\": str(model.dtype),\n                         \"max_pixels\": model.max_pixels, \"revision\": model.revision,\n                         \"timestamp\": datetime.now(timezone.utc).isoformat(timespec=\"seconds\")})\n        if i % shard_size == 0:\n            flush()\n        if i % log_every == 0 or i == len(todo):\n            el = time.time() - t0\n            print(f\"  {i}/{len(todo)} pairs | {el / (2 * i):.2f} s per forward | left ~{el / i * (len(todo) - i) / 60:.0f} min\", flush=True)\n    flush()\n", "src/mfrob/run/intervene.py": "\"\"\"Interventions: occlude the evidence region (target) and K matched control regions, score every variant.\"\"\"\nfrom __future__ import annotations\n\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\nfrom PIL import Image\n\nfrom ..data.release import SIDES, Record\nfrom ..geometry.controls import RADIAL_NEAREST, growth, seed_from, translate\nfrom ..geometry.supports import SUPPORTS, eccentricity, forbidden_zone, mask_hash, pair_union\nfrom ..models.base import VLM\nfrom ..operators.pixel import apply, blurred\nfrom .clean_pass import image_key, load_logits\n\n\nCONTROLS = (\"growth\", \"translate\", \"radial\")\nSCORE_FIELDS = [\"ld\", \"l_yes\", \"l_no\", \"logits_yes\", \"logits_no\", \"lse_all\", \"top1_id\", \"top1_is_yesno\", \"n_img_tokens\",\n                \"dtype\", \"max_pixels\", \"revision\", \"timestamp\"]\n\n\ndef build_regions(rec: Record, support: str = \"pair_union\", k: int = 4, margin: int = 8, control: str = \"growth\"):\n    \"\"\"(target mask, [control masks]) for a pair-fixed support, or (target, None) if no control fits.\n\n    A support that covers exactly the pixels of pair_union gets the controls of pair_union, so that the two\n    experiments differ only where the regions differ.\n    \"\"\"\n    target = SUPPORTS[support](rec)\n    union = pair_union(rec)\n    forb = forbidden_zone(target, union, margin)\n    name = \"pair_union\" if mask_hash(target) == mask_hash(union) else support\n    if control == \"growth\":\n        n = int(target.sum())\n        controls = [growth(forb, n, seed_from(rec.record_id, \"pair\", name, \"growth\", i)) for i in range(k)]\n        return target, (None if any(c is None for c in controls) else controls)\n    if control in (\"translate\", \"radial\"):\n        return target, translate(target, forb, k, seed_from(rec.record_id, \"pair\", name, control),\n                                 nearest=RADIAL_NEAREST if control == \"radial\" else None)\n    raise ValueError(f\"unknown control {control}\")\n\n\ndef load_cache(dirs) -> dict[str, dict]:\n    \"\"\"image_key -> saved score of exactly that image, from other experiments of the same model.\n    The key hashes the record, the side, the operator and the pixels of the region, so equal keys mean equal images.\"\"\"\n    cache = {}\n    for d in dirs:\n        rows = load_logits(d)\n        if len(rows):\n            for r in rows[[\"image_key\"] + SCORE_FIELDS].assign(reused_from=Path(d).name).to_dict(\"records\"):\n                cache[r.pop(\"image_key\")] = r\n    return cache\n\n\ndef infeasible_ids(out_dir: Path) -> set[str]:\n    return {l.strip() for f in Path(out_dir).glob(\"infeasible-*.txt\") for l in f.read_text().splitlines() if l.strip()}\n\n\ndef pending(records: list[Record], out_dir: Path) -> list[Record]:\n    \"\"\"Records with neither saved rows nor an infeasible mark.\"\"\"\n    done = load_logits(out_dir)\n    seen = (set(done.record_id) if len(done) else set()) | infeasible_ids(out_dir)\n    return [r for r in records if r.record_id not in seen]\n\n\ndef run_interventions(model: VLM, records: list[Record], out_dir: Path, operator: str = \"black\", support: str = \"pair_union\",\n                      k: int = 4, margin: int = 8, tag: str = \"\", shard_size: int = 100, log_every: int = 25,\n                      control: str = \"growth\", cache_dirs=()) -> None:\n    \"\"\"records: pairs with B = 1. Pairs without a feasible control are written to infeasible-*.txt and skipped.\n\n    cache_dirs: other experiments of the same model. An image that was already scored there (same record, side,\n    operator and region pixels) is not scored again; its row carries the column `reused_from`.\n    \"\"\"\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    todo = pending(records, out_dir)\n    print(f\"interventions [{operator}, {support}, {control}, K={k}]: {len(records)} pairs, {len(records) - len(todo)} done, {len(todo)} to go\")\n    cache = load_cache(cache_dirs)\n    n_reused = 0\n    rows, t0, n_fwd = [], time.time(), 0\n    n_parts = len(list(out_dir.glob(f\"part-{tag}*.parquet\")))\n\n    def flush():\n        nonlocal rows, n_parts\n        if rows:\n            pd.DataFrame(rows).to_parquet(out_dir / f\"part-{tag}{n_parts:05d}.parquet\", index=False)\n            n_parts += 1\n            rows = []\n\n    def mark_infeasible(rid):\n        with open(out_dir / f\"infeasible-{tag or 'all'}.txt\", \"a\") as f:\n            f.write(rid + \"\\n\")\n\n    def one(rec):\n        nonlocal rows, n_fwd, n_reused\n        target, controls = build_regions(rec, support, k, margin, control)\n        if controls is None:\n            mark_infeasible(rec.record_id)\n            return\n        regions = [(\"target\", -1, target)] + [(\"control\", j, c) for j, c in enumerate(controls)]\n        hashes = [mask_hash(m) for _, _, m in regions]\n        n_pix = int(target.sum())\n        assert all(int(m.sum()) == n_pix for _, _, m in regions)\n        for side in SIDES:\n            keys = [image_key(rec.record_id, side, operator, h) for h in hashes]\n            new = [i for i, key in enumerate(keys) if key not in cache]\n            scores = {}\n            if new:\n                clean = Image.open(rec.image_path[side])\n                blur = blurred(clean) if operator == \"blur\" else None\n                images = [apply(clean, regions[i][2], operator, seed_from(rec.record_id, side, operator, hashes[i]), blur) for i in new]\n                stamp = {\"dtype\": str(model.dtype), \"max_pixels\": model.max_pixels, \"revision\": model.revision,\n                         \"timestamp\": datetime.now(timezone.utc).isoformat(timespec=\"seconds\")}\n                scores = {i: {**sc, **stamp} for i, sc in zip(new, model.score(images, rec.question))}\n                n_fwd += len(new)\n            n_reused += len(regions) - len(new)\n            for i, (kind, j, m) in enumerate(regions):\n                rows.append({\"model\": model.name, \"record_id\": rec.record_id, \"side\": side, \"question_kind\": \"primary\",\n                             \"operator\": operator, \"support\": support, \"region_kind\": kind,\n                             \"control_type\": control if kind == \"control\" else None, \"k\": j, \"n_pixels\": n_pix,\n                             \"eccentricity\": eccentricity(m), \"image_key\": keys[i], **(scores[i] if i in scores else cache[keys[i]])})\n\n    for i, rec in enumerate(todo, 1):\n        one(rec)\n        if i % shard_size == 0:\n            flush()\n        if i % log_every == 0 or i == len(todo):\n            el = time.time() - t0\n            print(f\"  {i}/{len(todo)} pairs | {el / max(n_fwd, 1):.2f} s per forward | left ~{el / i * (len(todo) - i) / 60:.0f} min\", flush=True)\n    flush()\n    if cache:\n        print(f\"reused from earlier experiments: {n_reused} images, scored now: {n_fwd}\")\n", "scripts/analyze_clean.py": "#!/usr/bin/env python3\n\"\"\"Summary of a clean pass: pair accuracy, ordering inside the pair, error types, by category.\n\n    python scripts/analyze_clean.py --model qwen2vl_2b\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nfrom mfrob.data.release import CATEGORY_ORDER, SIDES, find_release_root, load_release  # noqa: E402\nfrom mfrob.run.clean_pass import load_logits  # noqa: E402\n\n\ndef pair_table(logits: pd.DataFrame, records) -> pd.DataFrame:\n    \"\"\"One row per pair: B (correct on both clean sides), ordering, error type.\"\"\"\n    d = logits[logits.question_kind == \"primary\"].drop_duplicates([\"record_id\", \"side\"], keep=\"last\")\n    w = d.pivot(index=\"record_id\", columns=\"side\", values=[\"ld\", \"top1_is_yesno\", \"n_img_tokens\"])\n    rows = []\n    for r in records:\n        if r.record_id not in w.index:\n            continue\n        ld = {s: float(w.loc[r.record_id, (\"ld\", s)]) for s in SIDES}\n        valid = all(bool(w.loc[r.record_id, (\"top1_is_yesno\", s)]) for s in SIDES)\n        yes_side = \"original\" if r.gold[\"original\"] == \"yes\" else \"edited\"\n        no_side = \"edited\" if yes_side == \"original\" else \"original\"\n        ld_yes, ld_no = ld[yes_side], ld[no_side]\n        rows.append({\"record_id\": r.record_id, \"cat\": r.cat, \"group_id\": r.group_id, \"ld_yes\": ld_yes, \"ld_no\": ld_no,\n                     \"B\": valid and ld_yes > 0 and ld_no <= 0, \"ordering_ok\": ld_yes > ld_no,\n                     \"yes_yes\": ld_yes > 0 and ld_no > 0, \"no_no\": ld_yes <= 0 and ld_no <= 0,\n                     \"valid_tokens\": valid, \"n_img_tokens\": int(w.loc[r.record_id, (\"n_img_tokens\", \"original\")]),\n                     \"strict_eligible\": r.strict_eligible})\n    return pd.DataFrame(rows)\n\n\ndef boot_ci(x, groups, n_boot=2000, seed=20260803):\n    \"\"\"Mean with a 95% cluster bootstrap interval (clusters = image groups).\"\"\"\n    x = np.asarray(x, float); codes, _ = pd.factorize(groups)\n    sums = np.bincount(codes, weights=x); cnt = np.bincount(codes)\n    rng = np.random.default_rng(seed); idx = rng.integers(0, len(cnt), (n_boot, len(cnt)))\n    m = sums[idx].sum(1) / cnt[idx].sum(1)\n    return float(x.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", required=True)\n    args = ap.parse_args()\n    run = Path(\"outputs/logits\") / args.model / \"clean\"\n    records = load_release(find_release_root(), with_sizes=False)\n    P = pair_table(load_logits(run), records)\n    cols = [\"B\", \"ordering_ok\", \"yes_yes\", \"no_no\"]\n    T = P.groupby(\"cat\")[cols].mean().reindex(CATEGORY_ORDER)\n    T.insert(0, \"pairs\", P.groupby(\"cat\").size().reindex(CATEGORY_ORDER))\n    T.loc[\"ALL\"] = [len(P)] + [P[c].mean() for c in cols]\n    T = T.rename(columns={\"B\": \"pair accuracy (PA)\", \"ordering_ok\": \"right ordering\", \"yes_yes\": \"yes on both\", \"no_no\": \"no on both\"})\n    pd.set_option(\"display.width\", 200)\n    print(T.round(3).to_string())\n    pa, order = boot_ci(P.B, P.group_id), boot_ci(P.ordering_ok, P.group_id)\n    print(f\"\\nPA {pa[0]:.3f} [{pa[1]:.3f}, {pa[2]:.3f}] | right ordering {order[0]:.3f} [{order[1]:.3f}, {order[2]:.3f}]\"\n          f\" | pairs with B = 1: {int(P.B.sum())} of {len(P)} | top-1 token is yes/no: {P.valid_tokens.mean():.4f}\")\n    out = Path(\"outputs/reports\"); out.mkdir(parents=True, exist_ok=True)\n    P.to_parquet(run / \"pairs.parquet\", index=False)\n    meta = json.loads((run / \"run_meta.json\").read_text())\n    (out / f\"clean_{args.model}.json\").write_text(json.dumps({\"run\": meta, \"pa\": pa, \"right_ordering\": order,\n        \"by_category\": T.round(4).reset_index().to_dict(orient=\"records\")}, ensure_ascii=False, indent=1))\n    print(\"saved\", run / \"pairs.parquet\", \"and\", out / f\"clean_{args.model}.json\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/analyze_er.py": "#!/usr/bin/env python3\n\"\"\"ER, ER_YES, ER_NO, ROA, LVF of one experiment.   python scripts/analyze_er.py --model qwen2vl_2b\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom analyze_clean import boot_ci, pair_table  # noqa: E402\nfrom mfrob.data.release import CATEGORY_ORDER, SIDES, find_release_root, load_release  # noqa: E402\nfrom mfrob.metrics.er import pair_er  # noqa: E402\nfrom mfrob.run.clean_pass import load_logits  # noqa: E402\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", required=True)\n    ap.add_argument(\"--experiment\", default=\"black__pair_union__growth_k4\")\n    args = ap.parse_args()\n    base = Path(\"outputs/logits\") / args.model\n    records = load_release(find_release_root(), with_sizes=False)\n    gold = {(r.record_id, s): r.gold[s] for r in records for s in SIDES}\n    P = pair_table(load_logits(base / \"clean\"), records).set_index(\"record_id\")\n    E = pair_er(load_logits(base / args.experiment), gold).set_index(\"record_id\")\n    E[\"cat\"], E[\"group_id\"] = P.cat.reindex(E.index), P.group_id.reindex(E.index)\n    infeasible = {l.strip() for f in (base / args.experiment).glob(\"infeasible-*.txt\") for l in f.read_text().splitlines() if l.strip()}\n    n_eligible = len(P) - len({r for r in P.index if r in infeasible}) - 0      # among clean-correct pairs only the tried ones are known\n    T = E.groupby(\"cat\")[[\"er\", \"er_yes\", \"er_no\", \"roa\"]].mean().reindex(CATEGORY_ORDER)\n    T.insert(0, \"pairs\", E.groupby(\"cat\").size().reindex(CATEGORY_ORDER))\n    T.loc[\"ALL\"] = [len(E)] + [E[c].mean() for c in (\"er\", \"er_yes\", \"er_no\", \"roa\")]\n    pd.set_option(\"display.width\", 200)\n    print(T.round(3).to_string())\n    ci = {c: boot_ci(E[c], E.group_id) for c in (\"er\", \"er_yes\", \"er_no\", \"roa\")}\n    print(\"\\n\" + \" | \".join(f\"{c} {v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]\" for c, v in ci.items()))\n    print(f\"pairs with B = 1: {int(P.B.sum())} | with a control: {len(E)} | without: {len(infeasible)} | K = {int(E.k.max())}\")\n    out = Path(\"outputs/reports\"); out.mkdir(parents=True, exist_ok=True)\n    E.reset_index().to_parquet(base / args.experiment / \"pair_er.parquet\", index=False)\n    (out / f\"er_{args.model}_{args.experiment}.json\").write_text(json.dumps(\n        {\"ci\": ci, \"by_category\": T.round(4).reset_index().to_dict(orient=\"records\"), \"infeasible\": len(infeasible)}, indent=1))\n\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/audit.py": "#!/usr/bin/env python3\n\"\"\"Checks the release against the facts listed in PLAN.md (stage 1). Exits with 1 on any mismatch.\n\n    python scripts/audit.py                # everything that the files on disk allow\n    python scripts/audit.py --meta-only    # annotations and ROI metadata only\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport sys\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nfrom mfrob.data.release import SIDES, find_release_root, load_release, read_checksums  # noqa: E402\n\nEXPECT_CATEGORIES = {\"add\": 212, \"remove\": 223, \"replace_appearance\": 256, \"color_alter\": 209, \"counting\": 150,\n                     \"relation\": 110, \"textual_change\": 222, \"surface_material_change\": 218}\nEXPECT_ABSENT = {(\"add\", \"original\"): 212, (\"remove\", \"edited\"): 223, (\"counting\", \"original\"): 1,\n                 (\"counting\", \"edited\"): 8, (\"textual_change\", \"original\"): 3, (\"textual_change\", \"edited\"): 1}\nEXPECT_SIZES = {(512, 512): 454, (1024, 1024): 263, (640, 480): 197}\n\nFAILED = []\n\n\ndef check(name: str, got, want=None) -> None:\n    ok = bool(got) if want is None else got == want\n    print(f\"[{'ok' if ok else 'FAIL'}] {name}: {got}\" + (\"\" if ok or want is None else f\"  (expected {want})\"))\n    if not ok:\n        FAILED.append(name)\n\n\ndef sha256(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--root\", type=Path, default=None)\n    ap.add_argument(\"--meta-only\", action=\"store_true\")\n    args = ap.parse_args()\n    root = args.root or find_release_root()\n    print(\"release:\", root)\n    have_images = not args.meta_only and (root / \"images\").is_dir()\n    have_masks = not args.meta_only and (root / \"segmentation\").is_dir()\n    recs = load_release(root, with_sizes=have_images)\n\n    # --- metadata\n    check(\"unique record ids\", len({r.record_id for r in recs}), 1600)\n    check(\"categories\", dict(Counter(r.category for r in recs)), EXPECT_CATEGORIES)\n    gold = Counter((r.gold[\"original\"], r.gold[\"edited\"]) for r in recs)\n    check(\"gold (original, edited)\", dict(gold), {(\"no\", \"yes\"): 1376, (\"yes\", \"no\"): 224})\n    yes_orig = Counter(r.category for r in recs if r.gold[\"original\"] == \"yes\")\n    check(\"categories with yes on the original side\", dict(yes_orig), {\"remove\": 223, \"textual_change\": 1})\n    check(\"strict eligible\", sum(r.strict_eligible for r in recs), 1591)\n    check(\"protocols\", dict(Counter(r.protocol for r in recs)),\n          {\"target_only\": 1538, \"target_plus_reference\": 53, \"attribute_specific\": 9})\n    check(\"target_plus_reference only in relation\",\n          {r.category for r in recs if r.protocol == \"target_plus_reference\"}, {\"relation\"})\n    absent = Counter((r.category, s) for r in recs for s in SIDES if r.evidence[s].absent)\n    check(\"absent sides\", dict(absent), EXPECT_ABSENT)\n    check(\"no record absent on both sides\", not any(all(r.evidence[s].absent for s in SIDES) for r in recs))\n    check(\"absent sides carry no evidence\",\n          all(e.mask_path is None and not e.boxes and e.area == 0 for r in recs for e in r.evidence.values() if e.absent))\n    check(\"present sides carry a mask, boxes and an area\",\n          all(e.mask_path is not None and e.boxes and e.bbox and e.area > 0\n              for r in recs for e in r.evidence.values() if not e.absent))\n    roi = {x[\"record_id\"]: x for x in json.loads((root / \"roi_annotations.json\").read_text())[\"records\"]}\n    check(\"csv envelope equals legacy_union_bbox_xyxy\",\n          all(list(r.envelope) == roi[r.record_id][\"legacy_union_bbox_xyxy\"] for r in recs))\n    check(\"distinct image groups\", len({r.group_id for r in recs}), 1584)\n    check(\"all questions end with 'Answer yes or no.'\", all(r.question.endswith(\"Answer yes or no.\") for r in recs))\n    check(\"every record has a non-flip question\", all(\"non_flip\" in r.aux_questions for r in recs))\n    manifest = json.loads((root / \"manifest.json\").read_text())\n    for name in (\"annotations.csv\", \"roi_annotations.json\"):\n        check(f\"sha256 of {name} matches the manifest\", sha256(root / name) == manifest[\"sha256\"][name])\n\n    # --- images\n    if have_images:\n        checksums = read_checksums(root)\n        bad = [str(p) for r in recs for p in r.image_path.values() if sha256(p) != checksums[str(p.relative_to(root))]]\n        check(\"sha256 of all 3,200 images\", len(bad), 0)\n        sizes = Counter(r.size for r in recs)\n        check(\"distinct image sizes\", len(sizes), 186)\n        check(\"most frequent sizes\", {k: sizes[k] for k in EXPECT_SIZES}, EXPECT_SIZES)\n        big = Counter(r.category for r in recs if r.size == (1024, 1024))\n        check(\"1024x1024 pairs by category\", dict(big), {\"counting\": 150, \"relation\": 110, \"remove\": 2, \"replace_appearance\": 1})\n\n        def inside(b, size):\n            return 0 <= b[0] < b[2] <= size[0] and 0 <= b[1] < b[3] <= size[1]\n        check(\"every box lies inside its image\",\n              all(inside(r.envelope, r.size) and all(inside(b, r.size) for e in r.evidence.values() for b in e.boxes) for r in recs))\n    else:\n        print(\"[skip] image checks: images are not on disk\")\n\n    # --- masks\n    if have_masks:\n        n, bad = 0, []\n        for r in recs:\n            for s in SIDES:\n                e = r.evidence[s]\n                if e.absent:\n                    continue\n                m = np.asarray(Image.open(e.mask_path).convert(\"L\")); n += 1\n                on = m > 127\n                ys, xs = np.nonzero(on)\n                env = (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)\n                if not (set(np.unique(m).tolist()) <= {0, 255} and m.shape == (r.size[1], r.size[0])\n                        and int(on.sum()) == e.area and env == e.bbox):\n                    bad.append(f\"{r.record_id}:{s}\")\n        check(\"evidence masks checked\", n, 2752)\n        check(\"masks: values 0/255, image size, area and envelope match the metadata\", len(bad), 0)\n        if bad:\n            print(\"   first offenders:\", bad[:5])\n    else:\n        print(\"[skip] mask checks: segmentation/ is not on disk\")\n\n    print(\"\\nAUDIT\", \"FAILED: \" + \", \".join(FAILED) if FAILED else \"PASSED\")\n    return 1 if FAILED else 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "scripts/build_geometry.py": "#!/usr/bin/env python3\n\"\"\"Feasibility of growth controls for the pair supports -> outputs/geometry/feasibility.csv\"\"\"\nfrom __future__ import annotations\n\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nfrom mfrob.data.release import CATEGORY_ORDER, find_release_root, load_release  # noqa: E402\nfrom mfrob.geometry.controls import growth_feasible  # noqa: E402\nfrom mfrob.geometry.supports import SUPPORTS, forbidden_zone, pair_union  # noqa: E402\n\nrows = []\nfor r in load_release(find_release_root()):\n    u = pair_union(r)\n    row = {\"record_id\": r.record_id, \"cat\": r.cat, \"strict_eligible\": r.strict_eligible, \"W\": r.size[0], \"H\": r.size[1]}\n    for name, fn in SUPPORTS.items():\n        s = fn(r)\n        row[f\"n_{name}\"] = int(s.sum())\n        row[f\"growth_{name}\"] = growth_feasible(forbidden_zone(s, u), int(s.sum()))\n    row[\"same_pixels\"] = row[\"n_pair_union\"] == row[\"n_pair_envelope\"] and bool((u == SUPPORTS[\"pair_envelope\"](r)).all())\n    rows.append(row)\nF = pd.DataFrame(rows)\nout = Path(\"outputs/geometry\"); out.mkdir(parents=True, exist_ok=True)\nF.to_csv(out / \"feasibility.csv\", index=False)\nT = F.groupby(\"cat\")[[\"growth_pair_union\", \"growth_pair_envelope\", \"same_pixels\"]].sum().reindex(CATEGORY_ORDER)\nT.loc[\"ALL\"] = T.sum()\nprint(T.astype(int).to_string())\n", "scripts/clean.py": "#!/usr/bin/env python3\n\"\"\"Clean pass of one model over the release.   python scripts/clean.py --model qwen2vl_2b [--limit 10]\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport sys\nfrom pathlib import Path\n\nimport yaml\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nfrom mfrob.data.release import find_release_root, load_release  # noqa: E402\nfrom mfrob.data.subset import subset  # noqa: E402\nfrom mfrob.models import make_model  # noqa: E402\nfrom mfrob.run.clean_pass import load_logits, run_clean_pass  # noqa: E402\n\n\ndef versions() -> dict:\n    import importlib.metadata as md\n    out = {}\n    for name in (\"torch\", \"transformers\", \"accelerate\", \"pillow\", \"numpy\"):\n        try:\n            out[name] = md.version(name)\n        except md.PackageNotFoundError:\n            out[name] = None\n    return out\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", required=True)\n    ap.add_argument(\"--limit\", type=int, default=None, help=\"first N pairs of every category (smoke test)\")\n    ap.add_argument(\"--out\", type=Path, default=None)\n    ap.add_argument(\"--shard\", default=None, help=\"i/n: take every n-th pair starting from i (one worker per GPU)\")\n    ap.add_argument(\"--categories\", nargs=\"+\", default=None, help=\"release category names to keep\")\n    ap.add_argument(\"--per-category\", type=int, default=None, help=\"fixed random subset: N pairs of every category that have a control\")\n    ap.add_argument(\"--subset-seed\", type=int, default=0)\n    ap.add_argument(\"--log-every\", type=int, default=50)\n    args = ap.parse_args()\n    cfg = yaml.safe_load(Path(f\"configs/models/{args.model}.yaml\").read_text())\n    records = load_release(find_release_root())\n    if args.categories:\n        records = [r for r in records if r.category in args.categories]\n    if args.per_category:\n        records = subset(records, args.per_category, args.subset_seed)\n    if args.limit:\n        seen, keep = {}, []\n        for r in records:\n            if seen.setdefault(r.category, 0) < args.limit:\n                keep.append(r); seen[r.category] += 1\n        records = keep\n    tag = \"\"\n    if args.shard:\n        i, n = (int(v) for v in args.shard.split(\"/\"))\n        records, tag = records[i::n], f\"s{i}-\"\n    out = args.out or Path(\"outputs/logits\") / cfg[\"name\"] / (\"clean_smoke\" if args.limit else \"clean\")\n    done = load_logits(out)\n    if (out / \"run_meta.json\").exists() and len(done) and {r.record_id for r in records} <= set(done.record_id):\n        print(f\"{out}: nothing left to do\")\n        return\n    model = make_model(cfg)\n    yes, no = model.yes_no_ids()\n    meta = {\"model\": model.name, \"hf_id\": model.hf_id, \"revision\": model.revision, \"device\": model.device,\n            \"dtype\": str(model.dtype), \"max_pixels\": model.max_pixels, \"yes_ids\": yes, \"no_ids\": no,\n            \"prompt_template\": model.prompt_template, \"per_category\": args.per_category, \"subset_seed\": args.subset_seed,\n            \"code\": os.environ.get(\"MFROB_CODE_VERSION\", \"unknown\"), \"packages\": versions()}\n    out.mkdir(parents=True, exist_ok=True)\n    (out / \"run_meta.json\").write_text(json.dumps(meta, ensure_ascii=False, indent=1))\n    print(json.dumps(meta, ensure_ascii=False)[:600])\n    run_clean_pass(model, records, out, tag=tag, log_every=args.log_every)\n\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/compare_experiments.py": "#!/usr/bin/env python3\n\"\"\"Intervention experiments of one model side by side, each against a reference experiment.\n\n    python scripts/compare_experiments.py --model qwen2vl_2b [--reference black__pair_union__growth_k4] [--experiments A B ...]\n\nWithout --experiments every finished experiment of the model is taken (smoke runs only with --smoke).\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom analyze_clean import boot_ci, pair_table  # noqa: E402\nfrom mfrob.data.release import CATEGORY_ORDER, SIDES, fill_conflict, find_release_root, load_release  # noqa: E402\nfrom mfrob.metrics.er import pair_er, sigmoid  # noqa: E402\nfrom mfrob.run.clean_pass import load_logits  # noqa: E402\n\n\ndef load_experiment(path: Path, gold: dict, P: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Per-pair table of one experiment with the clean-pass columns it is compared to.\"\"\"\n    E = pair_er(load_logits(path), gold).set_index(\"record_id\")\n    E[\"cat\"], E[\"group_id\"] = P.cat.reindex(E.index), P.group_id.reindex(E.index)\n    E[\"harm_yes\"] = sigmoid(P.ld_yes.reindex(E.index)) - E.p_control_yes        # what a control alone costs\n    E[\"harm_no\"] = sigmoid(-P.ld_no.reindex(E.index)) - E.p_control_no\n    return E\n\n\ndef overview(E: dict[str, pd.DataFrame]) -> pd.DataFrame:\n    \"\"\"Every experiment on its own pairs.\"\"\"\n    rows = {}\n    for name, e in E.items():\n        ci = boot_ci(e.er, e.group_id)\n        rows[name] = {\"pairs\": len(e), \"er\": ci[0], \"er_lo\": ci[1], \"er_hi\": ci[2], \"er_yes\": e.er_yes.mean(), \"er_no\": e.er_no.mean(),\n                      \"roa\": e.roa.mean(), \"wrong_target_yes\": e.wrong_target_yes.mean(), \"wrong_control_yes\": e.wrong_control_yes.mean(),\n                      \"harm_yes\": e.harm_yes.mean(), \"harm_no\": e.harm_no.mean()}\n    return pd.DataFrame(rows).T\n\n\ndef against_reference(E: dict[str, pd.DataFrame], ref: str) -> pd.DataFrame:\n    \"\"\"Paired differences to the reference on the pairs both experiments have; 95% cluster bootstrap intervals.\"\"\"\n    rows = {}\n    for name, e in E.items():\n        if name == ref:\n            continue\n        ids = e.index.intersection(E[ref].index)\n        a, b, grp = e.loc[ids], E[ref].loc[ids], e.group_id.loc[ids]\n        row = {\"common_pairs\": len(ids)}\n        for col in (\"er\", \"er_yes\", \"er_no\"):\n            m, lo, hi = boot_ci(a[col] - b[col], grp)\n            row.update({f\"d_{col}\": m, f\"d_{col}_lo\": lo, f\"d_{col}_hi\": hi})\n        row[\"spearman_er_yes\"] = a.er_yes.corr(b.er_yes, method=\"spearman\")\n        row[\"roa\"], row[\"roa_ref\"] = a.roa.mean(), b.roa.mean()\n        rows[name] = row\n    return pd.DataFrame(rows).T\n\n\ndef by_category(E: dict[str, pd.DataFrame], col: str, ref: str) -> tuple[pd.DataFrame, pd.Series, int]:\n    \"\"\"Category means of `col` on the pairs common to all experiments, and the rank correlation of the category order with the reference.\"\"\"\n    ids = None\n    for e in E.values():\n        ids = e.index if ids is None else ids.intersection(e.index)\n    T = pd.DataFrame({name: e.loc[ids].groupby(\"cat\")[col].mean() for name, e in E.items()}).reindex(CATEGORY_ORDER)\n    T.loc[\"ALL\"] = [e.loc[ids, col].mean() for e in E.values()]\n    T.insert(0, \"pairs\", list(E[ref].loc[ids].groupby(\"cat\").size().reindex(CATEGORY_ORDER)) + [len(ids)])\n    rho = T.drop(columns=\"pairs\").drop(index=\"ALL\").corr(method=\"spearman\")[ref]\n    return T, rho, len(ids)\n\n\ndef colour_conflict(E: dict[str, pd.DataFrame], conflict: set[str]) -> pd.DataFrame:\n    rows = {}\n    for name, e in E.items():\n        c = e[e.cat == \"color\"]\n        hit = c.index.isin(conflict)\n        rows[name] = {\"n_conflict\": int(hit.sum()), \"er_yes_conflict\": c.er_yes[hit].mean(), \"er_no_conflict\": c.er_no[hit].mean(),\n                      \"n_other\": int((~hit).sum()), \"er_yes_other\": c.er_yes[~hit].mean(), \"er_no_other\": c.er_no[~hit].mean()}\n    return pd.DataFrame(rows).T\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", required=True)\n    ap.add_argument(\"--reference\", default=\"black__pair_union__growth_k4\")\n    ap.add_argument(\"--experiments\", nargs=\"+\", default=None)\n    ap.add_argument(\"--smoke\", action=\"store_true\", help=\"compare the __smoke runs instead of the full ones\")\n    args = ap.parse_args()\n    base = Path(\"outputs/logits\") / args.model\n    ref = args.reference + (\"__smoke\" if args.smoke else \"\")\n    names = args.experiments or sorted(p.name for p in base.iterdir() if p.is_dir() and not p.name.startswith(\"clean\")\n                                       and p.name.endswith(\"__smoke\") == args.smoke and list(p.glob(\"part-*.parquet\")))\n    names = [ref] + [n for n in names if n != ref]\n    records = load_release(find_release_root(), with_sizes=False)\n    gold = {(r.record_id, s): r.gold[s] for r in records for s in SIDES}\n    P = pair_table(load_logits(base / (\"clean_smoke\" if args.smoke and (base / \"clean_smoke\").exists() else \"clean\")), records).set_index(\"record_id\")\n    E = {n: load_experiment(base / n, gold, P) for n in names}\n    short = {n: n.replace(\"__growth_k4\", \"\").replace(\"__smoke\", \"\") for n in names}\n    E = {short[n]: e for n, e in E.items()}\n    ref = short[ref]\n\n    pd.set_option(\"display.width\", 250); pd.set_option(\"display.max_columns\", 30)\n    A, D = overview(E), against_reference(E, ref)\n    print(\"== 1. Every experiment on its own pairs ==\")\n    print(\"wrong_*: share of yes sides answered 'no' under the target / under a control;  harm_*: p(gold | clean) - p(gold | control)\")\n    print(A.round(3).to_string())\n    print(f\"\\n== 2. Difference to {ref} on the pairs both have (95% interval) ==\")\n    print(D.round(3).to_string())\n    out = {\"reference\": ref, \"overview\": A.round(5).reset_index().to_dict(orient=\"records\"),\n           \"against_reference\": D.round(5).reset_index().to_dict(orient=\"records\")}\n    for col in (\"er\", \"er_yes\"):\n        T, rho, n = by_category(E, col, ref)\n        print(f\"\\n== 3. {col} by category on the {n} pairs common to all experiments ==\")\n        print(T.round(3).to_string())\n        print(\"rank correlation of the category order with the reference: \" + \", \".join(f\"{k} {v:.2f}\" for k, v in rho.items() if k != ref))\n        out[f\"{col}_by_category\"] = T.round(5).reset_index().to_dict(orient=\"records\")\n        out[f\"{col}_category_rank_corr\"] = {k: float(v) for k, v in rho.items()}\n    C = colour_conflict(E, {r.record_id for r in records if fill_conflict(r)})\n    print(\"\\n== 4. Colour questions about gray / silver / black (a flat fill can answer them) against the other colour questions ==\")\n    print(C.round(3).to_string())\n    out[\"colour_conflict\"] = C.round(5).reset_index().to_dict(orient=\"records\")\n    rep = Path(\"outputs/reports\"); rep.mkdir(parents=True, exist_ok=True)\n    path = rep / f\"compare_{args.model}{'_smoke' if args.smoke else ''}.json\"\n    path.write_text(json.dumps(out, indent=1))\n    print(\"\\nsaved\", path)\n\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/decompose.py": "#!/usr/bin/env python3\n\"\"\"The parts of ER for every model and experiment, the pre-specified hypotheses G1-G6 and the verdicts V1-V4.\n\n    python scripts/decompose.py --predict --model qwen25vl_3b        # after the clean pass, before any occlusion\n    python scripts/decompose.py --models qwen2vl_2b qwen25vl_3b qwen25vl_7b [--smoke]\n\nER = (margin - harm - leak) / 2 holds exactly for every pair; the script stops if it does not.\nThresholds and rules are read from configs/prereg.yaml; nothing is decided by hand here.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport yaml\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom analyze_clean import pair_table  # noqa: E402\nfrom mfrob.data.release import CATEGORY_ORDER, SIDES, find_release_root, load_release  # noqa: E402\nfrom mfrob.data.subset import has_control  # noqa: E402\nfrom mfrob.metrics.bootstrap import boot_macro, boot_mean, boot_ratio  # noqa: E402\nfrom mfrob.metrics.er import components, sigmoid  # noqa: E402\nfrom mfrob.run.clean_pass import load_logits  # noqa: E402\n\nSCALES = (\"prob\", \"logodds\", \"rel\", \"flip\")\nREPORTS = Path(\"outputs/reports\")\n\n\ndef label(name: str, ref: dict) -> str:\n    \"\"\"Short name of an experiment: what differs from the reference.\"\"\"\n    operator, support, control = name.replace(\"__smoke\", \"\").split(\"__\")\n    control, k = control.rsplit(\"_k\", 1)\n    diff = [v for v, r in ((operator, ref[\"operator\"]), (support, ref[\"support\"]), (control, ref[\"control\"])) if v != r]\n    if int(k) != ref[\"k\"]:\n        diff.append(f\"K={k}\")\n    return \"+\".join(diff) or \"reference\"\n\n\ndef load_model(model: str, records, gold, pre: dict, smoke: bool) -> dict:\n    \"\"\"Clean-pass table and the components of every finished experiment of one model.\"\"\"\n    base = Path(\"outputs/logits\") / model\n    P = pair_table(load_logits(base / \"clean\"), records).set_index(\"record_id\")\n    exps, raw = {}, {}\n    for d in sorted(base.iterdir()):\n        if not d.is_dir() or d.name.startswith(\"clean\") or d.name.endswith(\"__smoke\") != smoke or not list(d.glob(\"part-*.parquet\")):\n            continue\n        rows = load_logits(d)\n        c = components(rows, P, gold, ecc_gap=pre[\"hypotheses\"][\"G5\"][\"max_ecc_gap\"])\n        gap = float((c.er - (c.margin - c.harm - c.leak) / 2).abs().max())\n        if not gap < 1e-9:\n            raise SystemExit(f\"{model}/{d.name}: ER differs from (margin - harm - leak) / 2 by {gap:.2e}\")\n        c[\"cat\"], c[\"group_id\"] = P.cat.reindex(c.index), P.group_id.reindex(c.index)\n        exps[label(d.name, pre[\"reference\"])], raw[label(d.name, pre[\"reference\"])] = c, rows\n    return {\"model\": model, \"P\": P, \"exps\": exps, \"raw\": raw}\n\n\ndef predict(model: str, records, pre: dict) -> None:\n    \"\"\"G1: what the clean pass alone says ER will be. Written once, before any occlusion of the model.\"\"\"\n    base = Path(\"outputs/logits\") / model\n    P = pair_table(load_logits(base / \"clean\"), records).set_index(\"record_id\")\n    by_id = {r.record_id: r for r in records}\n    ok = [rid for rid in P.index[P.B] if has_control(by_id[rid])]\n    half = 0.5 * float((sigmoid(P.ld_yes[ok]) - sigmoid(P.ld_no[ok])).mean())\n    occluded = any(list(d.glob(\"part-*.parquet\")) for d in base.iterdir() if d.is_dir() and not d.name.startswith(\"clean\"))\n    path = REPORTS / f\"prediction_{model}.json\"\n    REPORTS.mkdir(parents=True, exist_ok=True)\n    if path.exists():\n        print(f\"{path} exists and is kept: {path.read_text()}\")\n        return\n    out = {\"model\": model, \"eligible_pairs\": len(ok), \"half_margin\": half,\n           \"er_predicted_low\": half / (1 + pre[\"hypotheses\"][\"G1\"][\"max_relative_gap\"]), \"er_predicted_high\": half,\n           \"occlusion_results_already_present\": occluded, \"created_utc\": datetime.now(timezone.utc).isoformat(timespec=\"seconds\")}\n    path.write_text(json.dumps(out, indent=1))\n    print(f\"prediction for {model}: ER between {out['er_predicted_low']:.3f} and {half:.3f} on {len(ok)} pairs\"\n          + (\" | NOT blind: occlusion results exist\" if occluded else \"\") + f\" -> {path}\")\n\n\ndef fmt(ci) -> str:\n    return f\"{ci[0]:.3f} [{ci[1]:.3f}, {ci[2]:.3f}]\"\n\n\ndef parts_table(runs: list[dict], nb: int) -> pd.DataFrame:\n    rows = {}\n    for r in runs:\n        c = r[\"exps\"].get(\"reference\")\n        if c is None:\n            continue\n        er, ner = boot_mean(c.er, c.group_id, nb), boot_ratio(c.margin - c.harm - c.leak, c.margin - c.harm, c.group_id, nb)\n        rows[r[\"model\"]] = {\"pairs\": len(c), \"PA\": float(r[\"P\"].B.mean()), \"ER\": er[0], \"ER_lo\": er[1], \"ER_hi\": er[2],\n                            \"half_margin\": c.margin.mean() / 2, \"half_harm\": c.harm.mean() / 2, \"half_leak\": c.leak.mean() / 2,\n                            \"nER\": ner[0], \"nER_lo\": ner[1], \"nER_hi\": ner[2], \"default\": c.default.mean(),\n                            \"PS\": c.ps.mean(), \"ROA\": c.roa.mean(), \"R2_ER_margin\": float(np.corrcoef(c.er, c.margin)[0, 1] ** 2)}\n    return pd.DataFrame(rows).T\n\n\ndef hypotheses(runs: list[dict], pre: dict, nb: int) -> list[dict]:\n    H, out = pre[\"hypotheses\"], []\n    blind = set(pre[\"models\"][\"blind\"]) | set(pre[\"models\"].get(\"fallback\", {}).values())\n    ref = {r[\"model\"]: r[\"exps\"][\"reference\"] for r in runs if \"reference\" in r[\"exps\"]}\n\n    # G1\n    rows, ok = [], []\n    for m, c in ref.items():\n        rel = (0.5 * c.margin.mean() - c.er.mean()) / c.er.mean()\n        note = \"\"\n        p = REPORTS / f\"prediction_{m}.json\"\n        if p.exists():\n            pr = json.loads(p.read_text())\n            raw = next(r[\"raw\"][\"reference\"] for r in runs if r[\"model\"] == m)\n            first = raw.timestamp[raw.reused_from.isna()].min() if \"reused_from\" in raw else raw.timestamp.min()\n            note = f\"predicted {pr['er_predicted_low']:.3f}..{pr['er_predicted_high']:.3f}, written {'before' if pr['created_utc'] <= first else 'AFTER'} the first occlusion\"\n        rows.append(f\"{m}{'' if m in blind else ' (seen)'}: ER {c.er.mean():.3f}, half margin {c.margin.mean() / 2:.3f}, relative gap {rel:.3f}\" + (f\" | {note}\" if note else \"\"))\n        if m in blind:\n            ok.append(c.er.mean() > 0 and abs(rel) <= H[\"G1\"][\"max_relative_gap\"])\n    out.append({\"id\": \"G1\", \"passed\": (all(ok) if ok else None), \"detail\": rows, \"rule\": f\"ER > 0 and |relative gap| <= {H['G1']['max_relative_gap']} for every blind model\"})\n\n    # G2\n    if len(ref) >= 2:\n        ids = set.intersection(*(set(c.index) for c in ref.values()))\n        own = (sorted(ref, key=lambda m: ref[m].er.mean()), sorted(ref, key=lambda m: ref[m].margin.mean()))\n        com = (sorted(ref, key=lambda m: ref[m].er[list(ids)].mean()), sorted(ref, key=lambda m: ref[m].margin[list(ids)].mean())) if ids else (None, None)\n        out.append({\"id\": \"G2\", \"passed\": own[0] == own[1] and com[0] == com[1] and bool(ids),\n                    \"detail\": [f\"own pairs: by ER {own[0]}, by margin {own[1]}\", f\"{len(ids)} common pairs: by ER {com[0]}, by margin {com[1]}\",\n                               \"own pairs, ER / half margin: \" + \", \".join(f\"{m} {c.er.mean():.3f} / {c.margin.mean() / 2:.3f}\" for m, c in ref.items())],\n                    \"rule\": \"identical order on own and on common pairs\"})\n    else:\n        out.append({\"id\": \"G2\", \"passed\": None, \"detail\": [\"needs at least two models\"], \"rule\": \"\"})\n\n    # G3\n    rows, ok = [], []\n    for a, b in itertools.combinations(ref, 2):\n        ids = ref[a].index.intersection(ref[b].index)\n        d_er, d_m = ref[a].er[ids] - ref[b].er[ids], ref[a].margin[ids] - ref[b].margin[ids]\n        if abs(d_er.mean()) >= H[\"G3\"][\"min_gap\"]:\n            share = boot_ratio(0.5 * d_m, d_er, ref[a].group_id[ids], nb)\n            ok.append(share[0] >= H[\"G3\"][\"min_share\"])\n            rows.append(f\"{a} vs {b}, {len(ids)} common pairs: ER gap {d_er.mean():+.3f}, share explained by the margin {fmt(share)}\")\n        else:\n            rows.append(f\"{a} vs {b}, {len(ids)} common pairs: ER gap {d_er.mean():+.3f} is below {H['G3']['min_gap']}, not used\")\n    out.append({\"id\": \"G3\", \"passed\": (all(ok) if ok else None), \"detail\": rows or [\"needs at least two models\"],\n                \"rule\": f\"share >= {H['G3']['min_share']} for every model pair with a gap of at least {H['G3']['min_gap']}\"})\n\n    # G4\n    loc = Path(\"outputs/geometry/edit_locality.parquet\")\n    rows, ok = [], []\n    if loc.exists():\n        L = pd.read_parquet(loc).set_index(\"record_id\")\n        for m, c in ref.items():\n            share = (c.ld_leak > H[\"G4\"][\"leak_logodds\"]).groupby(c.cat).mean().reindex(CATEGORY_ORDER).fillna(0).sort_values(ascending=False, kind=\"stable\")\n            rho = float(c.ld_leak.abs().corr(L.outside_mean_abs.reindex(c.index), method=\"spearman\"))\n            top = list(share.index[:2])\n            rows.append(f\"{m}{'' if m in blind else ' (seen)'}: leak pairs {(c.ld_leak > H['G4']['leak_logodds']).mean():.3f}; top categories {top} \"\n                        f\"({share.iloc[0]:.2f}, {share.iloc[1]:.2f}); Spearman with the change outside the region {rho:.2f}\")\n            if m in blind:\n                ok.append(set(top) == set(H[\"G4\"][\"top_categories\"]) and rho > H[\"G4\"][\"min_spearman\"])\n    else:\n        rows.append(f\"{loc} is missing: run scripts/edit_locality.py\")\n    out.append({\"id\": \"G4\", \"passed\": (all(ok) if ok else None), \"detail\": rows,\n                \"rule\": f\"for every blind model the top two categories are {H['G4']['top_categories']} and Spearman > {H['G4']['min_spearman']}\"})\n\n    # G5\n    rows, ok = [], []\n    for m, c in ref.items():\n        s = c[c.n_matched >= 1]\n        if len(s):\n            d = boot_mean(s.er_matched - s.er, s.group_id, nb)\n            ok.append(abs(d[0]) <= H[\"G5\"][\"max_er_shift\"])\n            rows.append(f\"{m}: {len(s)} of {len(c)} pairs have a control at the same distance to the centre; ER shift {fmt(d)}\")\n    out.append({\"id\": \"G5\", \"passed\": (all(ok) if ok else None), \"detail\": rows, \"rule\": f\"|ER shift| <= {H['G5']['max_er_shift']} for every model\"})\n\n    # G6\n    rows, ok = [], []\n    for r in runs:\n        for ctrl in H[\"G6\"][\"controls\"]:\n            if ctrl in r[\"exps\"] and \"reference\" in r[\"exps\"]:\n                v, c = r[\"exps\"][ctrl], r[\"exps\"][\"reference\"]\n                ids = v.index.intersection(c.index)\n                d = boot_mean(v.er[ids] - c.er[ids], c.group_id[ids], nb)\n                good = -pre[\"equivalence_margin_er\"] <= d[1] and d[2] <= pre[\"equivalence_margin_er\"] and v.ps[ids].mean() >= H[\"G6\"][\"min_ps\"]\n                ok.append(good)\n                rows.append(f\"{r['model']}, {ctrl}: {len(ids)} common pairs, ER difference {fmt(d)}, PS {v.ps[ids].mean():.3f}\")\n    out.append({\"id\": \"G6\", \"passed\": (all(ok) if ok else None), \"detail\": rows or [\"no translate / radial experiments yet\"],\n                \"rule\": f\"interval inside +-{pre['equivalence_margin_er']} and PS >= {H['G6']['min_ps']}\"})\n    return out\n\n\ndef scale_view(c: pd.DataFrame, scale: str) -> pd.DataFrame:\n    \"\"\"er, er_yes, er_no, roa and the confidence-adjusted er of one scale; rows where the scale is undefined are dropped.\"\"\"\n    s = \"prob\" if scale == \"macro\" else scale\n    v = pd.DataFrame({\"er\": c[f\"er_{s}\"], \"er_yes\": c[f\"er_yes_{s}\"], \"er_no\": c[f\"er_no_{s}\"],\n                      \"roa\": c.roa_logodds if s == \"logodds\" else c.roa, \"cat\": c.cat, \"group_id\": c.group_id})\n    v[\"adj\"] = {\"prob\": c.er_prob - c.margin / 2, \"logodds\": c.er_logodds - c.ld_margin / 2}.get(s, v.er)\n    return v.dropna(subset=[\"er\", \"er_yes\", \"er_no\"])\n\n\ndef mean_ci(x, v: pd.DataFrame, scale: str, nb: int):\n    return boot_macro(x, v.cat, v.group_id, nb) if scale == \"macro\" else boot_mean(x, v.group_id, nb)\n\n\ndef verdict_v2(v, ref_v, scale, V, nb, is_scale) -> str:\n    er = mean_ci(v.er, v, scale, nb)\n    roa = float(v.roa.groupby(v.cat).mean().mean()) if scale == \"macro\" else float(v.roa.mean())\n    if er[1] <= 0 or roa < V[\"fail_roa\"]:\n        return \"fails\"\n    ratio_ok = True if is_scale else er[0] >= V[\"min_ratio\"] * mean_ci(ref_v.er, ref_v, scale, 200)[0]\n    return \"holds\" if ratio_ok and roa >= V[\"min_roa\"] else \"weakens\"\n\n\ndef verdict_v3(v, scale, V, nb) -> str:\n    d = mean_ci(v.er_yes - v.er_no, v, scale, nb)\n    if d[1] <= 0:\n        return \"fails\"\n    yes, no = mean_ci(v.er_yes, v, scale, 200)[0], mean_ci(v.er_no, v, scale, 200)[0]\n    return \"holds\" if no <= V[\"max_no_to_yes\"] * yes else \"weakens\"\n\n\ndef verdict_pair(va, vb, scale, sign, nb) -> tuple[str, str]:\n    \"\"\"V1 and V4 for two models on the pairs both have.\"\"\"\n    ids = va.index.intersection(vb.index)\n    if len(ids) < 2:\n        return \"n/a\", \"n/a\"\n    a, b = va.loc[ids], vb.loc[ids]\n    d = mean_ci(a.er - b.er, a, scale, nb)\n    sig = d[1] > 0 or d[2] < 0\n    v1 = \"weakens\" if not sig else (\"holds\" if np.sign(d[0]) == sign else \"fails\")\n    if not sig:\n        return v1, \"fails\"\n    adj = mean_ci(a.adj - b.adj, a, scale, nb)\n    return v1, \"holds\" if (adj[1] > 0 or adj[2] < 0) else \"weakens\"\n\n\ndef verdicts(runs: list[dict], pre: dict, nb: int) -> pd.DataFrame:\n    V = pre[\"verdicts\"]\n    names = sorted({k for r in runs for k in r[\"exps\"] if k != \"reference\"})\n    variations = [(\"reference\", \"prob\")] + [(n, \"prob\") for n in names] + [(\"reference\", s) for s in (\"logodds\", \"rel\", \"flip\", \"macro\")]\n    pairs = list(itertools.combinations([r[\"model\"] for r in runs], 2))\n    by = {r[\"model\"]: r[\"exps\"] for r in runs}\n    sign = {}\n    for a, b in pairs:\n        if \"reference\" in by[a] and \"reference\" in by[b]:\n            ids = by[a][\"reference\"].index.intersection(by[b][\"reference\"].index)\n            sign[(a, b)] = np.sign((by[a][\"reference\"].er[ids] - by[b][\"reference\"].er[ids]).mean())\n    rows = {}\n    for exp, scale in variations:\n        row = {}\n        for m, exps in by.items():\n            if exp in exps and \"reference\" in exps:\n                v = scale_view(exps[exp], scale)\n                ids = v.index.intersection(exps[\"reference\"].index)\n                ref_v = scale_view(exps[\"reference\"], scale).reindex(ids).dropna(subset=[\"er\"])\n                vv = v.loc[ref_v.index]\n                row[f\"V2 {m}\"] = verdict_v2(vv, ref_v, scale, V[\"V2\"], nb, is_scale=scale != \"prob\")\n                row[f\"V3 {m}\"] = verdict_v3(vv, scale, V[\"V3\"], nb)\n        for a, b in pairs:\n            if exp in by[a] and exp in by[b] and (a, b) in sign:\n                row[f\"V1 {a}/{b}\"], row[f\"V4 {a}/{b}\"] = verdict_pair(scale_view(by[a][exp], scale), scale_view(by[b][exp], scale), scale, sign[(a, b)], nb)\n        rows[exp if scale == \"prob\" else f\"scale: {scale}\"] = row\n    T = pd.DataFrame(rows).T\n    return T[sorted(T.columns)].fillna(\"-\")\n\n\ndef scales_table(runs: list[dict], nb: int) -> pd.DataFrame:\n    rows = {}\n    for r in runs:\n        c = r[\"exps\"].get(\"reference\")\n        if c is None:\n            continue\n        for scale in SCALES + (\"macro\",):\n            v = scale_view(c, scale)\n            er = mean_ci(v.er, v, scale, nb)\n            rows[(r[\"model\"], scale)] = {\"pairs\": len(v), \"ER\": er[0], \"ER_lo\": er[1], \"ER_hi\": er[2],\n                                         \"ER_YES\": mean_ci(v.er_yes, v, scale, 200)[0], \"ER_NO\": mean_ci(v.er_no, v, scale, 200)[0],\n                                         \"ROA\": float(v.roa.mean())}\n    return pd.DataFrame(rows).T\n\n\ndef variations_table(runs: list[dict], nb: int) -> pd.DataFrame:\n    rows = {}\n    for r in runs:\n        ref = r[\"exps\"].get(\"reference\")\n        for name, c in r[\"exps\"].items():\n            if ref is None:\n                continue\n            ids = c.index.intersection(ref.index)\n            d = boot_mean(c.er[ids] - ref.er[ids], ref.group_id[ids], nb) if name != \"reference\" else (0.0, 0.0, 0.0)\n            ner = 1 - c.leak.sum() / (c.margin - c.harm).sum()\n            rows[(r[\"model\"], name)] = {\"pairs\": len(c), \"ER\": c.er.mean(), \"ER_YES\": c.er_yes.mean(), \"ER_NO\": c.er_no.mean(), \"ROA\": c.roa.mean(),\n                                        \"PS\": c.ps.mean(), \"half_harm\": c.harm.mean() / 2, \"half_leak\": c.leak.mean() / 2, \"nER\": ner,\n                                        \"common\": len(ids), \"dER\": d[0], \"dER_lo\": d[1], \"dER_hi\": d[2]}\n    return pd.DataFrame(rows).T\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--models\", nargs=\"+\", default=None)\n    ap.add_argument(\"--model\", default=None, help=\"with --predict\")\n    ap.add_argument(\"--predict\", action=\"store_true\")\n    ap.add_argument(\"--smoke\", action=\"store_true\", help=\"use the __smoke experiments\")\n    ap.add_argument(\"--n-boot\", type=int, default=None)\n    args = ap.parse_args()\n    pre = yaml.safe_load(Path(\"configs/prereg.yaml\").read_text())\n    nb = args.n_boot or pre[\"bootstrap\"][\"n\"]\n    if args.predict:\n        predict(args.model, load_release(find_release_root()), pre)\n        return\n    records = load_release(find_release_root(), with_sizes=False)\n    gold = {(r.record_id, s): r.gold[s] for r in records for s in SIDES}\n    runs = [load_model(m, records, gold, pre, args.smoke) for m in args.models if (Path(\"outputs/logits\") / m / \"clean\").exists()]\n    pd.set_option(\"display.width\", 250); pd.set_option(\"display.max_columns\", 40); pd.set_option(\"display.max_rows\", 200)\n\n    A = parts_table(runs, nb)\n    print(\"== 1. Parts of ER in the reference experiment:  ER = half_margin - half_harm - half_leak ==\")\n    print(\"nER: share of the discrimination that sits inside the region; default: P(yes) with the evidence hidden; PS: share of controls weaker than the target\")\n    print(A.round(3).to_string())\n    Hs = hypotheses(runs, pre, nb)\n    print(\"\\n== 2. Hypotheses fixed in configs/prereg.yaml ==\")\n    for h in Hs:\n        state = {True: \"PASSED\", False: \"NOT PASSED\", None: \"not evaluated\"}[h[\"passed\"]]\n        print(f\"{h['id']}: {state}\" + (f\"   (rule: {h['rule']})\" if h[\"rule\"] else \"\"))\n        for line in h[\"detail\"]:\n            print(\"     \" + line)\n    S = scales_table(runs, nb)\n    print(\"\\n== 3. Scales, reference experiment ==\")\n    print(S.round(3).to_string())\n    T = variations_table(runs, nb)\n    print(\"\\n== 4. Protocol changes: one factor at a time; dER is the difference to the reference on common pairs ==\")\n    print(T.round(3).to_string())\n    Vt = verdicts(runs, pre, nb)\n    print(\"\\n== 5. Verdicts for the conclusions of the paper (rules of PLAN.md, reference C) ==\")\n    print(\"V1 model order by ER, V2 ER is high and specific, V3 YES >> NO asymmetry, V4 equal accuracy but different ER\")\n    print(Vt.to_string())\n    REPORTS.mkdir(parents=True, exist_ok=True)\n    path = REPORTS / f\"decompose{'_smoke' if args.smoke else ''}.json\"\n    path.write_text(json.dumps({\"models\": [r[\"model\"] for r in runs], \"parts\": A.round(5).reset_index().to_dict(\"records\"), \"hypotheses\": Hs,\n                                \"scales\": S.round(5).reset_index().to_dict(\"records\"), \"variations\": T.round(5).reset_index().to_dict(\"records\"),\n                                \"verdicts\": Vt.reset_index().to_dict(\"records\")}, indent=1, default=str))\n    print(\"\\nsaved\", path)\n\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/edit_locality.py": "#!/usr/bin/env python3\n\"\"\"How far the edit of every pair reaches outside the annotated region -> outputs/geometry/edit_locality.parquet\n\nModel-independent; needs the images. A pixel counts as changed if any channel differs by more than THRESHOLD.\nThe region is pair_union dilated by the control margin, i.e. everything a control is kept away from.\n\n    python scripts/edit_locality.py [--force]\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nfrom mfrob.data.release import CATEGORY_ORDER, find_release_root, load_release  # noqa: E402\nfrom mfrob.geometry.supports import MARGIN, dilate, pair_union  # noqa: E402\n\nTHRESHOLD = 25\nOUT = Path(\"outputs/geometry/edit_locality.parquet\")\n\n\ndef locality(rec) -> dict:\n    a = np.asarray(Image.open(rec.image_path[\"original\"]).convert(\"RGB\")).astype(np.int16)\n    b = np.asarray(Image.open(rec.image_path[\"edited\"]).convert(\"RGB\")).astype(np.int16)\n    diff = np.abs(a - b).max(2)\n    region = dilate(pair_union(rec), MARGIN)\n    changed, outside = diff > THRESHOLD, ~region\n    return {\"record_id\": rec.record_id, \"cat\": rec.cat, \"region_share\": float(region.mean()),\n            \"changed_share\": float(changed.mean()),\n            \"outside_changed_frac\": float((changed & outside).sum() / max(int(outside.sum()), 1)),\n            \"outside_mean_abs\": float(diff[outside].mean()) if outside.any() else 0.0,\n            \"inside_mean_abs\": float(diff[region].mean()) if region.any() else 0.0}\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--force\", action=\"store_true\")\n    args = ap.parse_args()\n    if OUT.exists() and not args.force:\n        print(f\"{OUT} exists\")\n        return\n    L = pd.DataFrame([locality(r) for r in load_release(find_release_root())])\n    OUT.parent.mkdir(parents=True, exist_ok=True)\n    L.to_parquet(OUT, index=False)\n    T = L.groupby(\"cat\")[[\"region_share\", \"changed_share\", \"outside_changed_frac\", \"outside_mean_abs\"]].median().reindex(CATEGORY_ORDER)\n    T[\"outside_changed_over_1pct\"] = (L.outside_changed_frac > 0.01).groupby(L.cat).mean().reindex(CATEGORY_ORDER)\n    print(\"medians by category; the last column is the share of pairs with more than 1% of the outside pixels changed\")\n    print(T.round(3).to_string())\n    print(\"saved\", OUT)\n\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/intervene.py": "#!/usr/bin/env python3\n\"\"\"Reference experiment R0 and its variants: target occlusion + K growth controls on pairs with B = 1.\n\n    python scripts/intervene.py --model qwen2vl_2b [--operator black] [--support pair_union] [--control growth] [--k 4]\n                                [--shard 0/2] [--limit 5]\n\nImages already scored in another experiment of the model (same record, side, operator, region pixels) are reused.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport yaml\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom analyze_clean import pair_table  # noqa: E402\nfrom mfrob.data.release import find_release_root, load_release  # noqa: E402\nfrom mfrob.models import make_model  # noqa: E402\nfrom mfrob.run.clean_pass import load_logits  # noqa: E402\nfrom mfrob.run.intervene import CONTROLS, pending, run_interventions  # noqa: E402\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", required=True)\n    ap.add_argument(\"--operator\", default=\"black\")\n    ap.add_argument(\"--support\", default=\"pair_union\")\n    ap.add_argument(\"--k\", type=int, default=4)\n    ap.add_argument(\"--shard\", default=None)\n    ap.add_argument(\"--control\", default=\"growth\", choices=CONTROLS)\n    ap.add_argument(\"--limit\", type=int, default=None, help=\"first N clean-correct pairs of every category (smoke test)\")\n    args = ap.parse_args()\n    cfg = yaml.safe_load(Path(f\"configs/models/{args.model}.yaml\").read_text())\n    records = load_release(find_release_root())\n    P = pair_table(load_logits(Path(\"outputs/logits\") / cfg[\"name\"] / \"clean\"), records)\n    ok = set(P[P.B].record_id)\n    records = [r for r in records if r.record_id in ok]\n    print(f\"pairs with B = 1: {len(records)} of {len(P)}\")\n    if args.limit:\n        seen, keep = {}, []\n        for r in records:\n            if seen.setdefault(r.category, 0) < args.limit:\n                keep.append(r); seen[r.category] += 1\n        records = keep\n    tag = \"\"\n    if args.shard:\n        i, n = (int(v) for v in args.shard.split(\"/\"))\n        records, tag = records[i::n], f\"s{i}-\"\n    smoke = \"__smoke\" if args.limit else \"\"\n    base = Path(\"outputs/logits\") / cfg[\"name\"]\n    out = base / f\"{args.operator}__{args.support}__{args.control}_k{args.k}{smoke}\"\n    if not pending(records, out):\n        print(f\"{out.name}: nothing left to do\")\n        return\n    cache_dirs = [d for d in sorted(base.iterdir()) if d.is_dir() and d.name.startswith(f\"{args.operator}__\") and d != out]\n    run_interventions(make_model(cfg), records, out, operator=args.operator, support=args.support, k=args.k, tag=tag,\n                      control=args.control, cache_dirs=cache_dirs)\n\nif __name__ == \"__main__\":\n    main()\n", "configs/data.yaml": "# Where the MechFaithBench v1.9 release lives. The first existing path wins; if none exists,\n# annotations.csv is searched for recursively under the search roots.\nrelease_roots:\n  - data/MechFaithBench\n  - /kaggle/input/datasets/fluloeo/mechfaithbench/MechFaithBench_v1_9_final_2026-07-29\nsearch_roots:\n  - data\n  - /kaggle/input\nsubset_seed: 0\nbootstrap_seed: 20260803\n", "configs/models/qwen25vl_3b.yaml": "name: qwen25vl_3b\nhf_id: Qwen/Qwen2.5-VL-3B-Instruct\nfamily: qwen_vl\nmax_pixels: 409600        # 640 x 640, the setting of the reference runs\ndtype: auto               # cuda: bf16 on Ampere+, fp16 otherwise; mps and cpu: fp32\nbatch_size: 2\n", "configs/models/qwen25vl_7b.yaml": "name: qwen25vl_7b\nhf_id: Qwen/Qwen2.5-VL-7B-Instruct\nfamily: qwen_vl\nmax_pixels: 409600        # 640 x 640, the setting of the reference runs\ndtype: auto               # cuda: bf16 on Ampere+, fp16 otherwise; mps and cpu: fp32\nbatch_size: 2\n", "configs/models/qwen2vl_2b.yaml": "name: qwen2vl_2b\nhf_id: Qwen/Qwen2-VL-2B-Instruct\nfamily: qwen_vl\nmax_pixels: 409600        # 640 x 640; the paper does not state its resolution (PLAN.md, stage 2)\ndtype: auto               # cuda: bf16 on Ampere+, fp16 otherwise; mps and cpu: fp32\nbatch_size: 2\n", "configs/models/qwen2vl_2b_native.yaml": "name: qwen2vl_2b_native\nhf_id: Qwen/Qwen2-VL-2B-Instruct\nfamily: qwen_vl\nmax_pixels: 12845056      # processor default from the model card: 1024x1024 images stay at native size\ndtype: auto               # cuda: bf16 on Ampere+, fp16 otherwise; mps and cpu: fp32\nbatch_size: 2\n", "configs/models/qwen2vl_7b.yaml": "name: qwen2vl_7b\nhf_id: Qwen/Qwen2-VL-7B-Instruct\nfamily: qwen_vl\nmax_pixels: 409600        # 640 x 640, the setting of the reference runs\ndtype: auto               # cuda: bf16 on Ampere+, fp16 otherwise; mps and cpu: fp32\nbatch_size: 2\n", "configs/prereg.yaml": "# Rules of the quick multi-axis check, fixed before the Kaggle run (plan: docs/plan_quick_2026-10-13.md).\n# After the tag `prereg-quick-1` this file is not edited; any later change goes to DEVIATIONS.md with its reason.\n# scripts/decompose.py reads the thresholds from here and prints pass / fail without manual judgement.\nversion: 1\ndate: 2026-10-13\n\nreference: {operator: black, support: pair_union, control: growth, k: 4}\nsubset: {per_category: 50, seed: 0}\n\nmodels:\n  seen: [qwen2vl_2b]                  # its results on all pairs were known when the thresholds were chosen\n  blind: [qwen25vl_3b, qwen25vl_7b]   # never run before this file was committed\n  fallback: {qwen25vl_7b: qwen2vl_7b}  # used only if the first one fails to run; counts as blind for G1 and G4.\n                                      # Its clean answers on all pairs were seen in an earlier run (PA 0.856), its ER never.\n\n# Pair quantities, P = P(yes); Y / N = the side whose gold answer is yes / no; C = mean over the controls; T = target occluded.\n#   margin  = Y_clean - N_clean\n#   harm    = margin - (Y_C - N_C)\n#   leak    = Y_T - N_T\n#   default = (Y_T + N_T) / 2\n#   ER      = (margin - harm - leak) / 2          exact identity\n#   nER     = 1 - sum(leak) / sum(margin - harm)\n#   PS      = share of controls with (Y_Ck - N_Ck) > leak\n# \"Eligible pairs\" of a model: in the subset, B = 1, and a growth control exists for pair_union.\n\nequivalence_margin_er: 0.02   # the smallest gap between adjacent models in Table 1 of the paper (0.017 and 0.025)\n\nhypotheses:\n  G1:   # ER of a model is predicted from its clean pass\n    statistic: \"(0.5 * mean(margin) - mean(ER)) / mean(ER) on the eligible pairs of the model\"\n    pass: \"mean(ER) > 0 and |statistic| <= max_relative_gap for every blind model\"   # two-sided: ER above half the margin by more than the gap fails too\n    max_relative_gap: 0.25\n    note: \"0.5 * mean(margin) is written to outputs/reports/prediction_<model>.json before any occlusion run of that model\"\n  G2:   # the order of the models by ER equals their order by margin\n    pass: \"identical order on own eligible pairs and on the pairs eligible for all models\"\n  G3:   # the ER gap between models is explained by the margin\n    statistic: \"0.5 * d(mean margin) / d(mean ER) on the pairs eligible for both models\"\n    pass: \">= min_share for every pair of models with |d ER| >= min_gap; not applicable if there is no such pair\"\n    min_share: 0.75\n    min_gap: 0.02\n  G4:   # leak is concentrated in appearance and color and follows the changes outside the region\n    leak_pair: \"log-odds discrimination under the target occlusion > leak_logodds\"\n    leak_logodds: 1.0\n    top_categories: [appearance, color]\n    min_spearman: 0.2\n    pass: \"for every blind model: the two categories with the largest share of leak pairs are top_categories, and Spearman(|log-odds discrimination under target|, mean absolute pixel change outside the dilated region) > min_spearman\"\n  G5:   # the harm of a control does not depend on its distance to the image centre\n    statistic: \"mean over pairs of ER computed from the controls with |eccentricity(control) - eccentricity(target)| <= max_ecc_gap, minus ER from all controls; pairs with at least one such control\"\n    max_ecc_gap: 0.1\n    max_er_shift: 0.02\n    pass: \"|statistic| <= max_er_shift for every model\"\n  G6:   # another way of placing the control does not change ER\n    controls: [translate, radial]\n    statistic: \"mean(ER with the control) - mean(ER with growth) on the pairs eligible under both, K = 4\"\n    min_ps: 0.95\n    pass: \"for every model with stage 2 and every control: the 95% interval of the statistic lies inside +-equivalence_margin_er, and PS >= min_ps\"\n\n# Verdicts for the conclusions of the paper under each protocol change (PLAN.md, reference C).\n# Model comparisons use the pairs eligible for both models under the given change.\nverdicts:\n  V1: {holds: \"ER difference between the models has the sign of the reference and its interval excludes 0\", weakens: \"interval includes 0\", fails: \"opposite sign, interval excludes 0\"}\n  V2:\n    holds: \"mean ER >= min_ratio * reference ER on the same pairs and ROA >= min_roa\"\n    weakens: \"ER > 0 (interval) but one of the two conditions is not met\"\n    fails: \"interval of ER includes 0 or ROA < fail_roa\"\n    min_ratio: 0.8\n    min_roa: 0.90\n    fail_roa: 0.5\n    scales: \"on another scale the ratio to the reference has no meaning: holds if ER > 0 (interval) and ROA >= min_roa\"\n  V3: {holds: \"ER_YES - ER_NO > 0 (interval) and ER_NO <= max_no_to_yes * ER_YES\", weakens: \"difference > 0 (interval) but ER_NO is larger\", fails: \"interval of the difference includes 0\", max_no_to_yes: 0.25}\n  V4:\n    pairs: \"eligible for both models (clean-correct for both)\"\n    holds: \"ER difference excludes 0 and the difference of (ER - margin / 2) excludes 0 too\"\n    weakens: \"ER difference excludes 0, the difference of (ER - margin / 2) includes 0\"\n    fails: \"interval of the ER difference includes 0\"\n    scales: \"logodds: the adjustment is ER_logodds - (log-odds margin) / 2. rel and flip are already normalised by the clean answer (on clean-correct pairs the flip margin is 1 for every model), so their adjusted difference is the difference itself. macro: as prob, with category macro means.\"\n  roa_on_scales: \"prob, rel, flip, macro use the probability-scale ROA; logodds uses the log-odds ROA\"\n  reference_sign: \"for V1 the sign is that of the mean ER difference of the model pair in the reference experiment on the probability scale\"\n\nscales:   # axis C; ld_g = +ld on a yes side, -ld on a no side; p = sigmoid(ld_g)\n  prob: \"mean_k p(C_k) - p(T)\"\n  logodds: \"mean_k ld_g(C_k) - ld_g(T), ld clipped to [-20, 20]\"\n  rel: \"(mean_k p(C_k) - p(T)) / mean_k p(C_k); undefined where mean_k p(C_k) < 0.05, the number of such sides is reported\"\n  flip: \"1[T wrong] - mean_k 1[C_k wrong]\"\n  macro: \"prob scale, mean of the eight category means\"\n\nbootstrap: {n: 10000, seed: 20260803, cluster: group_id, level: 0.95}\n", "pyproject.toml": "[project]\nname = \"mfrob\"\nversion = \"0.1.0\"\ndescription = \"Robustness of the MechFaithBench behavioural protocol\"\nrequires-python = \">=3.11\"\n\n[tool.pytest.ini_options]\npythonpath = [\"src\"]\ntestpaths = [\"tests\"]\n\n[tool.black]\nline-length = 110\n\n[tool.ruff]\nline-length = 110\n"}

for rel, text in FILES.items():
    path = os.path.join(REPO, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
os.chdir(REPO)
os.environ["MFROB_CODE_VERSION"] = CODE_VERSION
print(len(FILES), "files written to", REPO, "| code version", CODE_VERSION)

37 files written to /kaggle/working/mfrob_quick | code version 33a23d3


## Чекпоинты и восстановление

Все результаты лежат в `outputs/` внутри копии репозитория. Архив обновляется раз в 15 минут и после каждого шага.
При старте ноутбук ищет прошлые результаты этого же ноутбука в `/kaggle/input`: архив или уже распакованную папку `logits`.

In [3]:
_last_zip = time.time()

def make_zip(note=""):
    global _last_zip
    tmp, n = ZIP_PATH + ".tmp", 0
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, names in os.walk("outputs"):
            for name in names:
                z.write(os.path.join(root, name)); n += 1
    os.replace(tmp, ZIP_PATH); _last_zip = time.time()
    print(f"[checkpoint {note}] {n} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")

def find_previous(top):
    # (number of result files, kind, path) for every archive of this notebook and every unpacked outputs folder of it
    found = []
    def add_zip(path):
        with zipfile.ZipFile(path) as z:
            found.append((sum(n.endswith(".parquet") for n in z.namelist()), "zip", path))
    if os.path.isfile(top):
        add_zip(top); return found
    for root, dirs, names in os.walk(top):
        for name in names:
            if name == ZIP_NAME:
                add_zip(os.path.join(root, name))
        if os.path.basename(root) == "outputs" and os.path.exists(os.path.join(root, "quick_marker.json")):
            found.append((len(glob.glob(os.path.join(root, "**", "*.parquet"), recursive=True)), "dir", root))
            dirs[:] = []
    return sorted(found)

def restore():
    if glob.glob("outputs/logits/**/*.parquet", recursive=True):
        print("outputs/ already has results -- continuing from them"); return
    found = find_previous(RESTORE_FROM or "/kaggle/input")
    if not found:
        print("no previous results of this notebook found -- starting from scratch")
    else:
        n, kind, path = found[-1]
        if kind == "zip":
            with zipfile.ZipFile(path) as z:
                z.extractall(".")
        else:
            shutil.copytree(path, "outputs", dirs_exist_ok=True)
        print(f"restored from {path} ({kind}):", len(glob.glob("outputs/logits/**/*.parquet", recursive=True)), "result files")
        for d in sorted(glob.glob("outputs/logits/*/*")):
            print("  ", os.path.relpath(d, "outputs/logits"), "|", len(glob.glob(d + "/part-*.parquet")), "parts")
    os.makedirs("outputs", exist_ok=True)
    marker = {"per_category": N_PER_CATEGORY, "code": CODE_VERSION}
    if os.path.exists("outputs/quick_marker.json"):
        old = json.load(open("outputs/quick_marker.json"))
        assert old["per_category"] == N_PER_CATEGORY, f"restored results use another subset: {old}"
    else:
        json.dump(marker, open("outputs/quick_marker.json", "w"))

def run_workers(script_args, name, n_proc):
    # n_proc = 2: one process per GPU, each on its own share of the pairs; n_proc = 1: one process that sees every GPU
    os.makedirs("outputs/logs", exist_ok=True)
    n = n_proc if (FAKE or N_GPUS >= n_proc) else 1
    procs = []
    for i in range(n):
        env = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
        args = list(script_args)
        if n > 1:
            env["CUDA_VISIBLE_DEVICES"] = str(i)
            args += ["--shard", f"{i}/{n}"]
        log = open(f"outputs/logs/{name}_gpu{i}.log", "w")
        procs.append((subprocess.Popen([sys.executable, "-u"] + args, stdout=log, stderr=subprocess.STDOUT, env=env), log))
    last = [""] * n
    def progress():
        for i in range(n):
            lines = [l for l in open(f"outputs/logs/{name}_gpu{i}.log", errors="ignore").read().splitlines()
                     if " pairs" in l or "nothing left" in l or "reused from" in l]
            if lines and lines[-1] != last[i]:
                last[i] = lines[-1]; print(f"gpu{i}: {lines[-1].strip()}")
    while any(p.poll() is None for p, _ in procs):
        time.sleep(2 if FAKE else 20)
        progress()
        if time.time() - _last_zip > CHECKPOINT_MINUTES * 60:
            make_zip("periodic")
    progress()
    for i, (p, log) in enumerate(procs):
        log.close()
        if p.returncode != 0:
            print(f"--- process {i} failed, end of its log:\n" + "".join(open(f"outputs/logs/{name}_gpu{i}.log", errors="ignore").readlines()[-25:]))
            raise RuntimeError(f"{name}: process {i} exited with code {p.returncode}")
    make_zip(name)

def run(script_args):
    r = subprocess.run([sys.executable] + script_args, capture_output=True, text=True)
    print(r.stdout[-30000:])
    if r.returncode != 0:
        print(r.stderr[-3000:]); raise RuntimeError(" ".join(script_args))

def forecast(model, n_proc):
    # speed from the clean pass of this session, then the time the two stages will take for this model
    speeds = [float(v) for f in glob.glob(f"outputs/logs/{model}_clean_gpu*.log") for v in re.findall(r"([\d.]+) s per forward", open(f, errors="ignore").read())]
    if not speeds:
        return
    import pandas as pd
    solved = int(pd.read_parquet(f"outputs/logits/{model}/clean/pairs.parquet").B.sum())
    s, n = speeds[-1], (n_proc if N_GPUS >= n_proc else 1)
    print(f"forecast for {model}: {s:.2f} s per image, {solved} solved pairs -> reference about {solved * 10 * s / n / 60:.0f} min"
          + (f", all variations about {solved * 78 * s / n / 60:.0f} min" if model in VARIATION_MODELS and "variations" in STAGES else ""))

restore()

no previous results of this notebook found -- starting from scratch


## Этап 1. Три модели в эталонных настройках

Для каждой модели: чистый проход по подвыборке → файл с предсказанием ER (гипотеза Г1) → закрашивание области и четырёх контролей.
Предсказание записывается до закрашивания и потом не меняется.

In [4]:
import yaml, traceback
READY, FAILED = [], []
FALLBACK = yaml.safe_load(open("configs/prereg.yaml"))["models"].get("fallback", {})

def core(model, n_proc):
    t0 = time.time()
    if not FAKE:
        from huggingface_hub import snapshot_download
        snapshot_download(yaml.safe_load(open(f"configs/models/{model}.yaml"))["hf_id"])
    run_workers(["scripts/clean.py", "--model", model, "--out", f"outputs/logits/{model}/clean",
                 "--per-category", str(N_PER_CATEGORY), "--log-every", "20"], f"{model}_clean", n_proc)
    run(["scripts/analyze_clean.py", "--model", model])
    run(["scripts/decompose.py", "--predict", "--model", model])
    forecast(model, n_proc)
    run_workers(["scripts/intervene.py", "--model", model], f"{model}_reference", n_proc)
    run(["scripts/analyze_er.py", "--model", model])
    print(f"{model}: {(time.time() - t0) / 60:.0f} min")

if "core" in STAGES:
    for model, n_proc in MODELS:
        for name in [model] + ([FALLBACK[model]] if model in FALLBACK else []):
            print(f"\n################ {name} ################")
            try:
                core(name, n_proc)
                READY.append(name)
                break
            except Exception:
                traceback.print_exc()
                FAILED.append(name)
                print(f"!!!!!!!! {name} FAILED; the notebook goes on without it")
    run(["scripts/edit_locality.py"])
else:
    READY = [m for m in [m for m, _ in MODELS] + list(FALLBACK.values()) if glob.glob(f"outputs/logits/{m}/black__pair_union__growth_k4/part-*.parquet")]
print("models with results:", READY, "| failed:", FAILED)


################ qwen2vl_2b ################


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

gpu0: 20/206 pairs | 0.29 s per forward | left ~2 min
gpu1: 20/205 pairs | 0.29 s per forward | left ~2 min
gpu0: 60/206 pairs | 0.27 s per forward | left ~1 min
gpu1: 60/205 pairs | 0.27 s per forward | left ~1 min
gpu0: 80/206 pairs | 0.29 s per forward | left ~1 min
gpu1: 80/205 pairs | 0.29 s per forward | left ~1 min
gpu0: 100/206 pairs | 0.31 s per forward | left ~1 min
gpu1: 100/205 pairs | 0.31 s per forward | left ~1 min
gpu0: 120/206 pairs | 0.32 s per forward | left ~1 min
gpu1: 120/205 pairs | 0.32 s per forward | left ~1 min
gpu0: 160/206 pairs | 0.34 s per forward | left ~1 min
gpu1: 160/205 pairs | 0.34 s per forward | left ~1 min
gpu0: 206/206 pairs | 0.32 s per forward | left ~0 min
gpu1: 200/205 pairs | 0.33 s per forward | left ~0 min
gpu1: 205/205 pairs | 0.33 s per forward | left ~0 min
[checkpoint qwen2vl_2b_clean] 8 files -> /kaggle/working/mfrob_quick_outputs.zip (0.1 MB)
            pairs  pair accuracy (PA)  right ordering  yes on both  no on both
cat         

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

gpu0: 20/206 pairs | 0.36 s per forward | left ~2 min
gpu1: 20/205 pairs | 0.37 s per forward | left ~2 min
gpu0: 40/206 pairs | 0.37 s per forward | left ~2 min
gpu1: 40/205 pairs | 0.37 s per forward | left ~2 min
gpu0: 60/206 pairs | 0.38 s per forward | left ~2 min
gpu1: 60/205 pairs | 0.39 s per forward | left ~2 min
gpu0: 80/206 pairs | 0.41 s per forward | left ~2 min
gpu1: 80/205 pairs | 0.42 s per forward | left ~2 min
gpu0: 100/206 pairs | 0.43 s per forward | left ~2 min
gpu1: 100/205 pairs | 0.44 s per forward | left ~2 min
gpu1: 120/205 pairs | 0.44 s per forward | left ~1 min
gpu1: 140/205 pairs | 0.46 s per forward | left ~1 min
gpu1: 160/205 pairs | 0.46 s per forward | left ~1 min
gpu1: 180/205 pairs | 0.45 s per forward | left ~0 min
gpu1: 205/205 pairs | 0.44 s per forward | left ~0 min
--- process 0 failed, end of its log:
Loading weights: 100%|██████████| 824/824 [00:07<00:00, 114.65it/s]
{"model": "qwen25vl_3b", "hf_id": "Qwen/Qwen2.5-VL-3B-Instruct", "revision": 

Traceback (most recent call last):
  File "/tmp/ipykernel_23/3549521762.py", line 24, in <cell line: 0>
    core(name, n_proc)
    ~~~~^^^^^^^^^^^^^^
  File "/tmp/ipykernel_23/3549521762.py", line 10, in core
    run_workers(["scripts/clean.py", "--model", model, "--out", f"outputs/logits/{model}/clean",
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                 "--per-category", str(N_PER_CATEGORY), "--log-every", "20"], f"{model}_clean", n_proc)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_23/1479990622.py", line 84, in run_workers
    raise RuntimeError(f"{name}: process {i} exited with code {p.returncode}")
RuntimeError: qwen25vl_3b_clean: process 0 exited with code 1


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

gpu0: clean pass: 411 pairs, 0 already done, 411 to go
gpu0: 20/411 pairs | 0.56 s per forward | left ~7 min
gpu0: 40/411 pairs | 0.55 s per forward | left ~7 min
gpu0: 60/411 pairs | 0.56 s per forward | left ~7 min
gpu0: 80/411 pairs | 0.57 s per forward | left ~6 min
gpu0: 100/411 pairs | 0.58 s per forward | left ~6 min
gpu0: 120/411 pairs | 0.59 s per forward | left ~6 min
gpu0: 140/411 pairs | 0.60 s per forward | left ~5 min
gpu0: 160/411 pairs | 0.64 s per forward | left ~5 min
gpu0: 180/411 pairs | 0.67 s per forward | left ~5 min
gpu0: 200/411 pairs | 0.68 s per forward | left ~5 min
gpu0: 220/411 pairs | 0.68 s per forward | left ~4 min
gpu0: 240/411 pairs | 0.69 s per forward | left ~4 min
gpu0: 260/411 pairs | 0.71 s per forward | left ~4 min
gpu0: 280/411 pairs | 0.73 s per forward | left ~3 min
[checkpoint periodic] 27 files -> /kaggle/working/mfrob_quick_outputs.zip (0.5 MB)
gpu0: 300/411 pairs | 0.72 s per forward | left ~3 min
gpu0: 320/411 pairs | 0.72 s per forward 

### Итог первого этапа

**Таблица 1** — части ER для каждой модели: `ER = half_margin − half_harm − half_leak` (половины запаса, вреда контроля и утечки).
`nER` — доля различения пары, которая находится внутри размеченной области. `default` — вероятность «да», когда область закрашена.

**Раздел 2** — гипотезы Г1–Г6 (в выводе G1–G6) с отметкой PASSED / NOT PASSED. После первого этапа Г6 ещё не оценивается.

**Таблица 3** — те же выводы в других шкалах подсчёта. **Таблица 5** — вердикты для выводов статьи V1–V4.

In [5]:
if "core" in STAGES and READY:
    run(["scripts/decompose.py", "--models"] + READY)

== 1. Parts of ER in the reference experiment:  ER = half_margin - half_harm - half_leak ==
nER: share of the discrimination that sits inside the region; default: P(yes) with the evidence hidden; PS: share of controls weaker than the target
             pairs     PA     ER  ER_lo  ER_hi  half_margin  half_harm  half_leak    nER  nER_lo  nER_hi  default     PS    ROA  R2_ER_margin
qwen2vl_2b   313.0  0.762  0.325  0.310  0.340        0.360      0.016      0.019  0.946   0.921   0.969    0.163  0.981  0.971         0.626
qwen25vl_7b  355.0  0.864  0.411  0.397  0.424        0.456      0.014      0.030  0.932   0.907   0.954    0.112  0.987  0.983         0.352

== 2. Hypotheses fixed in configs/prereg.yaml ==
G1: PASSED   (rule: ER > 0 and |relative gap| <= 0.25 for every blind model)
     qwen2vl_2b (seen): ER 0.325, half margin 0.360, relative gap 0.107 | predicted 0.288..0.360, written before the first occlusion
     qwen25vl_7b: ER 0.411, half margin 0.456, relative gap 0.108 | predi

## Этап 2. Изменения протокола

Для двух меньших моделей по одному изменению за раз: чем закрашивать, что закрашивать, куда ставить контроль и сколько контролей.
Картинки, которые уже считались (например, закрашенная область при новом контроле), берутся из готовых результатов.

In [6]:
if "variations" in STAGES:
    procs = dict(MODELS)
    for model in [m for m in VARIATION_MODELS if m in READY]:
        print(f"\n################ {model}: variations ################")
        t0 = time.time()
        for operator, support, control, k in VARIATIONS:
            name = f"{operator}__{support}__{control}_k{k}"
            print(f"---- {name}")
            run_workers(["scripts/intervene.py", "--model", model, "--operator", operator, "--support", support,
                         "--control", control, "--k", str(k)], f"{model}_{name}", procs[model])
        print(f"{model}: {(time.time() - t0) / 60:.0f} min")


################ qwen2vl_2b: variations ################
---- gray__pair_union__growth_k4
gpu0: interventions [gray, pair_union, growth, K=4]: 157 pairs, 0 done, 157 to go
gpu1: interventions [gray, pair_union, growth, K=4]: 156 pairs, 0 done, 156 to go
gpu0: 25/157 pairs | 0.29 s per forward | left ~6 min
gpu1: 25/156 pairs | 0.31 s per forward | left ~7 min
gpu0: 50/157 pairs | 0.30 s per forward | left ~5 min
gpu1: 50/156 pairs | 0.30 s per forward | left ~5 min
gpu0: 75/157 pairs | 0.32 s per forward | left ~4 min
gpu1: 75/156 pairs | 0.33 s per forward | left ~4 min
gpu0: 100/157 pairs | 0.37 s per forward | left ~3 min
gpu1: 100/156 pairs | 0.36 s per forward | left ~3 min
gpu1: 125/156 pairs | 0.35 s per forward | left ~2 min
gpu0: 125/157 pairs | 0.36 s per forward | left ~2 min
gpu0: 150/157 pairs | 0.34 s per forward | left ~0 min
gpu1: 150/156 pairs | 0.34 s per forward | left ~0 min
gpu0: 157/157 pairs | 0.34 s per forward | left ~0 min
gpu1: 156/156 pairs | 0.34 s per for

## Итог

Полный отчёт по обоим этапам. **Таблица 4** — каждое изменение протокола против эталона: `dER` — разница ER на общих парах с 95% интервалом.
**Таблица 5** — вердикты: `holds` — вывод статьи сохраняется, `weakens` — ослабевает, `fails` — не сохраняется.
Строки — изменения протокола и шкалы; столбцы — вывод и модель (или пара моделей).

In [7]:
if READY and "variations" in STAGES:
    run(["scripts/decompose.py", "--models"] + READY)
if FAILED:
    print("!!!!!!!! models that failed and are missing above:", FAILED)
make_zip("final")
print(sorted(glob.glob("outputs/reports/*")))
if IS_KAGGLE:
    from IPython.display import FileLink, display
    os.chdir(BASE); display(FileLink(os.path.basename(ZIP_PATH)))

== 1. Parts of ER in the reference experiment:  ER = half_margin - half_harm - half_leak ==
nER: share of the discrimination that sits inside the region; default: P(yes) with the evidence hidden; PS: share of controls weaker than the target
             pairs     PA     ER  ER_lo  ER_hi  half_margin  half_harm  half_leak    nER  nER_lo  nER_hi  default     PS    ROA  R2_ER_margin
qwen2vl_2b   313.0  0.762  0.325  0.310  0.340        0.360      0.016      0.019  0.946   0.921   0.969    0.163  0.981  0.971         0.626
qwen25vl_7b  355.0  0.864  0.411  0.397  0.424        0.456      0.014      0.030  0.932   0.907   0.954    0.112  0.987  0.983         0.352

== 2. Hypotheses fixed in configs/prereg.yaml ==
G1: PASSED   (rule: ER > 0 and |relative gap| <= 0.25 for every blind model)
     qwen2vl_2b (seen): ER 0.325, half margin 0.360, relative gap 0.107 | predicted 0.288..0.360, written before the first occlusion
     qwen25vl_7b: ER 0.411, half margin 0.456, relative gap 0.108 | predi

/kaggle/working/mfrob_quick_outputs.zip